# Suite DIA — The Python3, Matlab, Excel, Latex, C, Cpp and SystemVerilog dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules (modeled over JavaScript values here);
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. Latex writes notation, and has no evaluator. C evaluates by C's
rules over its arithmetic types (and Cpp, which shares its kinds, by C++'s), and SystemVerilog by IEEE 1800's over
4-state vectors and reals. Each dialect's install adds its union of its kinds, named plainly, to the framework's
`Term`, as the branch named after it, and its terms are written in a store through its writers; it reads only its own
terms. What every dialect shares is suite FRM; the Patterns dialect is suites EXP and EVL. Reading Python
source (`parse`) and NumPy are the Python implementation's only.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as Patterns from "@mbse/expressions/Dialects/Patterns";
import * as CDialect from "@mbse/expressions/Dialects/C";
import * as CppDialect from "@mbse/expressions/Dialects/Cpp";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python3 from "@mbse/expressions/Dialects/Python3";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import * as TypeScript5 from "@mbse/expressions/Dialects/TypeScript5";
import { Domains as D, Errors as FE, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Repr, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { AttributeError, KeyError, NotImplementedError, ValueError } = Errors;
const { ImportError, IndexError, NameError, OverflowError, ZeroDivisionError } = FE;

const [P, M, X] = [Python3.Expressions, Matlab.Expressions, Excel.Expressions];
const [PV, MV, XV] = [Python3.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [PD, MD, XD] = [Python3.Domains, Matlab.Domains, Excel.Domains];
const [C, CV, CD] = [CDialect.Expressions, CDialect.Evaluators, CDialect.Domains];
const [SV, SVV, SVD] = [SystemVerilog.Expressions, SystemVerilog.Evaluators, SystemVerilog.Domains];
const [TS, TV, TSText] = [TypeScript5.Expressions, TypeScript5.Evaluators, TypeScript5.Text];
const [PText, MText, XText, LText, CText, SVText] = [Python3.Text, Matlab.Text, Excel.Text, Latex.Text, CDialect.Text, SystemVerilog.Text];

const store: any = new Builders.Store(); // every dialect's kinds, in one store
for (const install of [Framework.install, Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install,
  CDialect.install, CppDialect.install, SystemVerilog.install, TypeScript5.install]) {
  store.transaction(install);
}
const Term: any = store.registered("Expressions.Framework.Term");

/** The term value `spec` writes, a native value as `dialect`'s literal, built in a transaction. */
function term(spec: unknown, dialect: F.Dialect): any {
  return store.transaction(() => store.Expressions.Framework.Term(dialect.spec(spec)));
}

/** A dialect's terms, from their specs. */
const termsOf = (dialect: F.Dialect) => (spec: unknown): any => term(spec, dialect);
const [py, mat, xl, tex, cc, sv] = [termsOf(P.DIALECT), termsOf(M.DIALECT), termsOf(X.DIALECT), termsOf(Latex.Expressions.DIALECT),
  termsOf(C.DIALECT), termsOf(SV.DIALECT)];

/** Whether a term reads back the same from JSON and from YAML. */
function roundTrips(value: unknown): boolean {
  return ([[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]).every(
    ([encode, decode]) => F.same(decode(Term, encode(Term, value)), value));
}

/** The data the expressions read: people, and objects with lists. */
function data(txn: any): any {
  const S = txn.Schemas;
  const ints = () => S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt]));
  return txn.uid("dia.data/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Squad: S.OfObject, Board: S.OfObject, Bus: S.OfObject })
    .spec([
      (_: any) => _.Person.name("Person").ref(true).properties({ name: S.OfStr, age: S.OfInt, email: S.OfStr }),
      (_: any) => _.Squad.name("Squad").ref(true).properties({ ranks: ints() }),
      (_: any) => _.Board.name("Board").ref(true).properties({ name: S.OfStr, grid: S.OfContainer(
        (c: any) => c.index([S.OfInt]).element([S.OfContainer((d: any) => d.index([S.OfInt]).element([S.OfInt]))])) }),
      (_: any) => _.Bus.name("Bus").ref(true).properties({ name: S.OfStr, ports: ints() }),
    ])]))]);
}

store.transaction(data);
const [ann, bob, squad, board, bus] = store.transaction(() => [
  store.Person((p: any) => p.name("Ann").age(30n).email("ann@example.com")),
  store.Person((p: any) => p.name("Bob").age(70n)), // no email
  store.Squad((s: any) => s.ranks([1n, 2n, 3n])), // a list property is an array of its values
  store.Board((b: any) => b.name("b").grid([(e: any) => e.as_container([(x: any) => x.as_scalar(1n), (x: any) => x.as_scalar(2n)]),
    (e: any) => e.as_container([(x: any) => x.as_scalar(3n)])])),
  store.Bus((b: any) => b.name("b").ports([4n, 5n])),
]);

<a id="DIA-01"></a>
## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
{
  const [t, a, b, c, d, x, y, z] = [..."tabcdxyz"].map((n) => P.name(n));
  for (const [built, source] of [ // written with constructors, as source
    [P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
      "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18n),
      P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), "np"),
    "import numpy as np\nfrom numpy.ma import getmaskarray\n"
      + "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"],
    [P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1n), P.unaryop("not", P.attribute(a, "c")))),
      "(lambda a: a.b > 1 and not a.c)(t)"],
    [P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2n))), "x if a < b else -y ** 2"],
    [P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"],
    [P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"],
    [P.binop("-", P.binop("-", P.binop("**", 2n, P.unaryop("-", 1n)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"],
    [P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"],
    [P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"],
    [P.boolop("and", a, b, c), "a and b and c"], [P.boolop("and", P.boolop("and", a, b), c), "(a and b) and c"], // n values
    [P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"],
    [P.compare("<", a, b, ["<=", c], ["is not", d]), "a < b <= c is not d"], // a chain
    [P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1n), new Uint8Array([0]), true, "it's"),
      "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"],
    [P.call("f", a, P.starred(b), P.keyword("k", 1n), P.keyword(null, c)), "f(a, *b, k=1, **c)"], // keywords, unpacked
    [P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1n, "g")), "c")),
      "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"],
    [P.call("all", P.generatorexp(P.compare(">", P.attribute(P.name("p"), "pin"), 0n), P.comprehension("p", P.attribute(t, "ports")))),
      "all(p.pin > 0 for p in t.ports)"],
    [P.call("sum", P.generatorexp(1n, P.comprehension("p", x, [P.compare("in", P.name("p"), y), P.compare("not in", P.name("p"), z)]))),
      "sum(1 for p in x if p in y if p not in z)"],
    [P.binop("+", P.subscript(x, P.binop("-", a, 1n)), P.call("len", P.call("set", x))), "x[a - 1] + len(set(x))"],
    [P.call("max", P.generatorexp(P.ifexp(b, P.name("v"), z), P.comprehension("v", P.ifexp(a, x, y), [P.boolop("or", a, b)])), 1n),
      "max((v if b else z for v in (x if a else y) if a or b), 1)"],
    [P.subscript(t, P.tuple_(P.slice_(1n, null, 2n), P.slice_(), P.constant(P.ELLIPSIS), P.constant(null), P.starred(x))),
      "t[1::2, :, ..., None, *x]"], // a tensor's indices: slices, ..., a new axis, unpacked
    // every other kind: displays, comprehensions, lambdas, named expressions, await and yield, `@`, constants
    [P.listcomp(x, P.comprehension("x", y, [x]), P.comprehension("z", x)), "[x for x in y if x for z in x]"],
    [P.dictcomp(P.name("k"), P.name("v"), P.comprehension(P.tuple_(P.name("k"), P.name("v")), P.call(P.attribute(d, "items")))),
      "{k: v for k, v in d.items()}"],
    [P.setcomp(x, P.comprehension("x", y)), "{x for x in y}"], [P.listcomp(x, P.comprehension("x", y, [], true)), "[x async for x in y]"],
    [P.set_(1n, P.starred(a)), "{1, *a}"],
    [P.dict_([null, a], ["k", 1n]), "{**a, 'k': 1}"], [P.tuple_(), "()"], [P.tuple_(1n), "(1,)"], [P.tuple_(1n, 2n), "(1, 2)"],
    [P.subscript(x, P.tuple_()), "x[()]"], [P.subscript(x, P.tuple_(1n)), "x[1,]"], [P.subscript(x, P.slice_()), "x[:]"],
    [P.lambda_(0n), "lambda: 0"], [P.lambda_(P.name("k"), [], { kwonlyargs: ["k"] }), "lambda *, k: k"],
    [P.lambda_(a, ["b", P.arg("c", 1n)], { posonlyargs: ["a"], vararg: "d", kwonlyargs: ["e", P.arg("f", 2n)], kwarg: "g" }),
      "lambda a, /, b, c=1, *d, e, f=2, **g: a"],
    [P.namedexpr("y", 10n), "(y := 10)"], [P.await_(x), "await x"], [P.yield_(), "(yield)"], [P.yield_(x), "(yield x)"],
    [P.yieldfrom(x), "(yield from x)"], [P.binop("@", a, b), "a @ b"], [P.compare("is", a, b), "a is b"],
    [P.unaryop("-", P.binop("**", x, 2n)), "-x ** 2"], [P.attribute(1n, "real"), "(1).real"],
    [P.constant(new P.Complex(0, 2)), "2j"], [P.constant(null), "None"], [P.constant(P.ELLIPSIS), "..."],
    [P.concatenatedstring("a", "b"), "'a' 'b'"],
    [P.call("f", P.generatorexp(x, P.comprehension("x", y))), "f(x for x in y)"],
    [P.call("f", P.generatorexp(x, P.comprehension("x", y)), 1n), "f((x for x in y), 1)"],
  ] as [unknown, string][]) {
    const written = py(built);
    assert(PText.ToText(written) === source && roundTrips(written), PText.ToText(written));
  }
  for (const [built, source] of [ // printed as written, what a parser keeps beside its terms: an f-string's text and fields, parentheses
    [P.joinedstr(["a{{b}}", P.formattedvalue(x, "r", [">", P.formattedvalue(P.name("w"))]), " ", P.formattedvalue(y, null, null, true)]),
      "f'a{{b}}{x!r:>{w}} {y=}'"],
    [P.templatestr(["x", P.interpolation(y), "z"]), "t'x{y}z'"], [P.joinedstr([P.formattedvalue(P.set_(1n))], "rf", "\""), "rf\"{ {1}}\""],
    [P.concatenatedstring(P.constant("a", "'a'"), P.constant("b", "\"b\"")), "'a' \"b\""],
    [P.parenthesized(P.binop("+", a, b)), "(a + b)"], [P.binop("*", P.parenthesized(a), b), "(a) * b"],
    [P.dictcomp(x, null, P.comprehension("x", y)), "{**x for x in y}"], // Python 3.15's, which 3.14 does not parse
    [P.listcomp(P.starred(x), P.comprehension("x", y)), "[*x for x in y]"],
  ] as [unknown, string][]) {
    assert(PText.ToText(py(built)) === source && roundTrips(py(built)), PText.ToText(py(built)));
  }
  assert(!("FromText" in PText)); // reading Python source needs Python's parser: the Python implementation's only
  assert(same([NaN, Infinity, -Infinity, -2n].map((v) => PText.ToText(py(P.constant(v)))), [
    "float('nan')", "float('inf')", "-float('inf')", "-2"]));
  assert(same([null, P.ELLIPSIS, new P.Complex(0, 2), new P.Complex(1, -2), new P.Complex(1, 2.5), new P.Complex(0, -2.5)].map(
    (v) => PText.ToText(py(P.constant(v)))), ["None", "...", "2j", "(1-2j)", "(1+2.5j)", "-2.5j"])); // Python3's own
  assert(PText.ToText(py(P.binop("**", P.constant(new P.Complex(0, -2.5)), 2n))) === "(-2.5j) ** 2"); // its precedence, as its spelling's
  assert(PText.ToText(py(P.binop("**", -2n, 2n))) === "(-2) ** 2" && PText.ToText(py(P.unaryop("-", P.binop("**", 2n, 2n)))) === "-2 ** 2");
  assert(PText.ToText(py(P.call("np.ma.getmaskarray", P.name("x")))) === "np.ma.getmaskarray(x)"); // a dotted function
  assert(PText.ToText(py(P.parenthesized(P.binop("+", a, b)))) === "(a + b)" && PText.ToText(py(P.binop("*", P.parenthesized(a), b))) === "(a) * b");
  assert(PText.ToText(py(P.slice_(a, b))) === "a:b" && PText.ToText(py(P.tuple_(a))) === "(a,)" && PText.ToText(py(P.yield_(a))) === "(yield a)"); // alone
  assert(PText.ToText(py(P.call("f", P.namedexpr("y", 1n)))) === "f((y := 1))" && PText.ToText(py(P.starred(P.binop("+", a, b)))) === "*a + b");
  assert(PText.ToText(py(P.starred(P.compare("<", a, b)))) === "*(a < b)"); // unpacked: a bitwise or, or what binds tighter
  assert(PText.ToText(py(P.attribute(P.binop("+", 1n, 2n), "real"))) === "(1 + 2).real");
  const counted = py(P.generatorexp(P.name("p"), P.comprehension("p", P.name("ps"), [P.name("q")]))); // p is bound in its clause and the element
  assert(same([...SY.free(counted, P.DIALECT)].sort(), ["ps", "q"]) && P.DIALECT.validate(counted, ["ps", "q"]).length === 0);
  assert(same(P.DIALECT.validate(py(P.generatorexp(1n, P.comprehension("1p", P.name("ps")))), ["ps"]), [
    "generators 0 target: a name must be an identifier, got '1p'"]));
  assert(same(P.DIALECT.validate(py(P.generatorexp(1n, P.comprehension("p", P.name("p"))))), ["generators 0 iter: variable 'p' is not bound"]));
  const chain = py(P.compare("<", a, b, ["<=", c])); // one comparison chain: b written once, the operators of its links
  assert(PText.ToText(chain) === "a < b <= c" && same(F.form(chain).arguments.map((operand) => F.nameOf(operand)), ["a", "b", "c"]));
  assert(same((F.form(chain).attributes.get("comparisons") as F.Part[]).map((part) => part.attributes.get("op")), ["<", "<="]));
  raises(ValueError, () => PText.ToText(py(P.binop("+", 1n, P.import_("math", 2n)))), "an import can only enclose the whole expression");
  // Displays: a list display (the framework's container constructor), and a dict display, evaluated as Python's.
  assert(PText.ToText(py(P.list_(1n, x))) === "[1, x]" && PText.ToText(py(P.dict_(["a", 1n], [x, P.list_()]))) === "{'a': 1, x: []}"
    && PText.ToText(py(P.dict_())) === "{}");
  assert(equal(PV.OfAny(py(P.dict_(["a", 1n], ["a", 2n], ["b", P.list_(x)])), { x: 3n }), new Map<unknown, unknown>([["a", 2n], ["b", [3n]]]))); // a later key replaces an earlier's
  const kinds = P.DIALECT.kinds(); // an import binds the names its kind reads from its attributes
  assert(same(kinds.get(P.IMPORT)!.names!(F.form(py(P.import_("numpy.ma", 1n))).attributes), ["numpy"]));
  assert(same(kinds.get(P.IMPORTFROM)!.names!(F.form(py(P.importfrom("m", "n", 1n, "k"))).attributes), ["k"]));
  assert(equal(SY.imports(py(P.import_("numpy", P.importfrom("m", "n", 1n), "np")), P.DIALECT).map((i) => F.form(i).attributes),
    map([{ module: "numpy", alias: "np" }, { module: "m", name: "n" }])));
  assert(same(P.DIALECT.validate(py(P.import_("not a module", 1n))), ["an import's module must be a dotted name, got 'not a module'"]));
  assert(same(P.DIALECT.validate(py(P.import_("m", 1n, "a b"))), ["an import's alias must be an identifier, got 'a b'"]));
  assert(same(P.DIALECT.validate(py(P.importfrom("m.", "x y", 1n))), ["an importfrom's module must be a dotted name, got 'm.'",
    "an importfrom's name must be an identifier, got 'x y'"]));
  assert(same(P.DIALECT.validate(py(P.importfrom("m", "n", 1n, ""))), ["an importfrom needs an alias"])); // empty, not absent
  assert(same(P.DIALECT.validate(py(P.let_("a b", 1n, 2n))), ["a let's name must be an identifier, got 'a b'"]));
  assert(same(P.DIALECT.validate(py(P.attribute(P.name("x"), "1")), ["x"]), ["an attribute must be an identifier, got '1'"]));
  for (const [spec, problems] of [
    [P.importfrom("m", "n", P.name("n")), []],
    [(u: any) => u[P.IMPORT]((r: any) => r.module(7n).body(P.constant(1n))), ["an import's module must be a str, got int"]],
    [(u: any) => u[P.IMPORT]((r: any) => r.module("numpy")), ["an import needs a body"]],
    [(u: any) => u[P.LITERAL]((r: any) => r), ["a literal needs a value"]],
    [(u: any) => u[P.COMPARE]((r: any) => r.left(P.constant(1n))), ["a compare needs a comparison"]],
  ] as [unknown, string[]][]) {
    assert(same(P.DIALECT.validate(py(spec)), problems), Repr.repr(P.DIALECT.validate(py(spec))));
  }
  assert(P.DIALECT.validate(py(P.binop("@", 1n, 2n)), [], true).length === 0 && P.DIALECT.validate(py(P.compare("is", 1n, 2n)), [], true).length === 0);
  assert(same(P.DIALECT.validate(py(P.binop("@@", 1n, 2n)), [], true), ["'@@' is not a core operation"]));
  assert(same(P.DIALECT.validate(py(P.compare("<", 1n, 2n, ["===", 3n])), [], true), ["'===' is not a core operation"]));
  assert(P.DIALECT.validate(py(P.compare("<", 1n, 2n, ["===", 3n]))).length === 0); // an extension, unless the core is asked for
  assert(P.DIALECT.validate(py(P.binop("+", 1n, 2n)), [], true).length === 0);
  assert(P.DIALECT.validate(py(P.call("hasattr", P.name("x"), "k")), ["x"]).length === 0); // builtins are ambient
  assert(P.DIALECT.infer(py(P.binop("+", 1n, 2.5))) === PD.Float && P.DIALECT.infer(py(P.binop("+", true, 1n))) === PD.Int);
  assert(P.DIALECT.infer(py(P.binop("+", "a", "b"))) === PD.Str && P.DIALECT.infer(py(P.unaryop("-", 2.0))) === PD.Float);
  assert(P.DIALECT.infer(py(P.binop("**", 2n, 3n))).name() === "int | float" && P.DIALECT.infer(py(P.unaryop("not", 1n))) === PD.Bool);
  assert(P.DIALECT.infer(py(P.compare("<", 1n, 2.5))) === PD.Bool && P.DIALECT.infer(py(P.attribute(P.name("x"), "a")), { x: D.Anything }) === D.Anything);
  assert(P.DIALECT.infer(py(P.boolop("and", 1n, "a"))).name() === "int | str"); // Python's and gives an operand
  assert(P.DIALECT.infer(py(P.let_("a", 1n, P.binop("*", P.name("a"), 2.5)))) === PD.Float
    && P.DIALECT.infer(py(P.call("f", 1n)), { f: D.Anything }) === D.Anything);
  assert(P.DIALECT.infer(py(P.generatorexp(P.name("p"), P.comprehension("p", P.name("v")))), { v: D.Anything }) === D.Anything);
  assert(P.DIALECT.infer(py(P.compare("<", 1n, 2n, ["<=", 3.5]))) === PD.Bool); // a chain
  assert(P.DIALECT.infer(py(P.boolop("and", 1n, "a", 2.5))).name() === "int | str | float");
  assert(P.DIALECT.infer(py(P.compare("<", 1n, 2n, ["===", 3n]))) === D.Anything); // an extension's operator: not known
  assert(P.DIALECT.infer(py(P.constant(null))) === PD.NoneType && P.DIALECT.infer(py(P.constant(P.ELLIPSIS))) === PD.Ellipsis
    && P.DIALECT.infer(py(P.constant(new P.Complex(0, 2)))) === PD.Complex); // Python3's constants
  assert(P.DIALECT.infer(py(P.joinedstr([P.formattedvalue(x)])), { x: D.Anything }) === PD.Str
    && P.DIALECT.infer(py(P.subscript(x, P.slice_(1n))), { x: D.Anything }) === D.Anything);
  raises(TypeError, () => P.DIALECT.infer(py(P.compare("<", "a", 1n))), "< cannot take (str, int)");
  raises(TypeError, () => P.DIALECT.infer(py(P.compare("<", 1n, 2n, ["<=", "a"]))), "<= cannot take (int, str)");
  assert(PD.of(new Uint8Array()) === PD.Bytes && PD.Numeric.includes(PD.Bool));
  raises(TypeError, () => P.DIALECT.spec([1n]), "expected a term, a writer or a native value, got [1]"); // no Python3 constant holds a list
  raises(ValueError, () => PV.OfAny(term(Patterns.Expressions.operation("add", 1n, 2n), Patterns.Expressions.DIALECT)),
    "a Patterns call is not a Python3 term"); // a term of another dialect: Python3 reads its own
}

<a id="DIA-02"></a>
## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}, options: any = {}) =>
    PV.OfAny(py(expression), new PV.Scope(variables, options));
  const [a, t] = [P.name("a"), P.name("t")];
  assert(ev(P.boolop("or", 0n, "x")) === "x" && ev(P.boolop("and", "", 1n)) === "" && ev(P.boolop("and", 1n, 2n)) === 2n); // an operand, not a bool
  assert(ev(P.compare("==", 1n, 1.0)) === true && ev(P.binop("+", true, 1n)) === 2n);
  assert(ev(P.binop("-", P.binop("%", P.binop("//", 7n, 2n), 3n), P.binop("/", P.binop("**", 2n, 2n), 4n))) === -1.0);
  assert(ev(P.ifexp(P.compare(">", a, 1n), a, P.name("missing")), { a: 2n }) === 2n
    && ev(P.let_("a", 3n, P.binop("*", a, a))) === 9n);
  assert(ev(P.attribute(t, "age"), { t: ann }) === 30n && ev(P.call("hasattr", t, "email"), { t: bob }) === false); // properties are attributes
  assert(ev(P.call("getattr", t, "name"), { t: ann }) === "Ann" && ev(P.subscript(t, "x"), { t: { x: 1n } }) === 1n);
  assert(ev(P.call("hasattr", t, "_secret"), { t: ann }) === false && ev(P.call("hasattr", t, "email"), { t: ann }) === true);
  assert(ev(P.binop("+", P.binop("+", P.call("abs", P.unaryop("-", 2n)), P.call("len", "abc")), P.call("max", 1n, 2n))) === 7n
    && ev(P.call("f", 2n), { f: (v: bigint) => v + 1n }) === 3n);
  raises(AttributeError, () => ev(P.attribute(t, "email"), { t: bob }), "'Person' object has no attribute 'email'");
  raises(AttributeError, () => ev(P.attribute(t, "__class__"), { t: ann }), "attribute '__class__' is private");
  raises(NameError, () => ev(P.name("x")), "name 'x' is not defined");
  raises(NameError, () => ev(P.call("len", "a"), {}, { builtins: {} }), "name 'len' is not defined");
  function shout(value: string): string {
    return value.toUpperCase();
  }
  raises(TypeError, () => ev(P.call(P.subscript(t, "f"), "a"), { t: { f: shout } }),
    "calling 'shout' is not allowed by the scope"); // only what the scope provides, not what values hold
  raises(TypeError, () => ev(P.call(t), { t: 3n }), "calling 'int' is not allowed by the scope");

  // NumPy is Python's; a module of the scope stands in for it here.
  const masked = { masked: true };
  const numpy = { numpy: new PV.Module("numpy", {
    ma: new PV.Module("numpy.ma", { masked }), pi: Math.PI, _core: new PV.Module("numpy._core", {}) }) };
  assert(ev(P.import_("numpy.ma", P.attribute(P.attribute(P.name("numpy"), "ma"), "masked")), {}, { modules: numpy }) === masked); // binds numpy
  assert(ev(P.import_("numpy.ma", P.attribute(P.name("ma"), "masked"), "ma"), {}, { modules: numpy }) === masked
    && ev(P.import_("numpy", P.attribute(P.name("numpy"), "pi")), {}, { modules: numpy }) === Math.PI);
  raises(ImportError, () => ev(P.import_("numpy", 1n, "np")), "import of 'numpy' is not allowed by the scope"); // nothing is imported unless the scope allows it
  raises(ImportError, () => ev(P.import_("os", P.call("os.getcwd")), {}, { modules: numpy }), "import of 'os' is not allowed by the scope");
  raises(ImportError, () => ev(P.import_("numpy.nope", 1n), {}, { modules: numpy }), "No module named 'numpy.nope'");
  raises(ImportError, () => ev(P.import_("numpy._core", 1n), {}, { modules: numpy }), "No module named 'numpy._core'");
  raises(ImportError, () => ev(P.importfrom("numpy", "nope", 1n), {}, { modules: numpy }), "cannot import name 'nope' from 'numpy'");
  raises(ImportError, () => ev(P.importfrom("numpy", "_core", 1n), {}, { modules: numpy }), "cannot import name '_core' from 'numpy'");
  assert(PV.OfAny(py(P.binop("+", P.name("x"), 1n)), { x: 1n }) === 2n); // an object is the variables of a scope
  const adult = store.transaction(() => store.Expressions.Framework.Expression( // an expression, its root written by a writer
    (e: any) => e.root(P.compare(">=", P.attribute(P.name("t"), "age"), 18n))));
  assert(PV.OfAny(adult, { t: ann }) === true && PText.ToText(adult) === "t.age >= 18");

  // Python's rules, case by case. The Python suite runs this table against Python itself; here it runs against the
  // model of them.
  const n = P.name;
  const bytes = (...values: number[]) => new Uint8Array(values);
  const modules = { math: new PV.Module("math", { floor: (v: number) => BigInt(Math.floor(v)) }), ...numpy };
  const VALUES: [unknown, Record<string, unknown>, unknown][] = [ // (expression, variables, value): the value and its type
    [P.unaryop("not", n("v")), { v: null }, true], [P.unaryop("not", 0.0), {}, true], [P.unaryop("not", ""), {}, true],
    [P.unaryop("not", bytes()), {}, true], [P.unaryop("not", n("v")), { v: [] }, true], [P.unaryop("not", n("v")), { v: {} }, true],
    [P.unaryop("not", NaN), {}, false], [P.unaryop("not", n("v")), { v: { k: 1n } }, false],
    [P.compare("<", 1n, 1.5), {}, true], [P.compare(">", 2n, 1.5), {}, true], [P.compare("==", 1n, 1.5), {}, false],
    [P.compare("==", P.binop("+", P.binop("**", 2n, 60n), 1n), P.call("float", P.binop("**", 2n, 60n))), {}, false],
    [P.compare("<", 1n, Infinity), {}, true], [P.compare(">", 1n, -Infinity), {}, true], [P.compare("<", 1n, NaN), {}, false],
    [P.compare("<", 1.5, 2n), {}, true], [P.compare("==", NaN, NaN), {}, false], [P.compare("==", 3n, 3.0), {}, true],
    [P.compare("<", 3n, 3.5), {}, true], [P.compare("<", 2.5, 1.5), {}, false], [P.compare("<", true, 2n), {}, true],
    [P.compare("<", "a", "b"), {}, true], [P.compare("<", bytes(97), bytes(98)), {}, true], [P.compare("==", bytes(97), bytes(97)), {}, true],
    [P.compare("!=", "a", 1n), {}, true],
    [P.binop("+", "ab", "c"), {}, "abc"], [P.binop("+", bytes(97), bytes(98)), {}, bytes(97, 98)], [P.binop("*", "ab", 2n), {}, "abab"],
    [P.binop("*", 2n, bytes(120)), {}, bytes(120, 120)], [P.binop("*", "ab", -1n), {}, ""], [P.binop("//", -7n, 2n), {}, -4n],
    [P.binop("%", -7n, 2n), {}, 1n], [P.binop("%", 7n, -2n), {}, -1n], [P.binop("%", -7.5, 2n), {}, 0.5], [P.binop("%", 7.5, 2n), {}, 1.5],
    [P.binop("//", 7.5, 2n), {}, 3.0], [P.binop("**", 2n, -1n), {}, 0.5], [P.binop("**", 2.0, 3n), {}, 8.0],
    [P.binop("+", 1.5, 1n), {}, 2.5], [P.binop("*", 1.5, 2n), {}, 3.0], [P.binop("-", 3n, 1.5), {}, 1.5], [P.binop("/", 1n, 2n), {}, 0.5],
    [P.unaryop("-", true), {}, -1n], [P.unaryop("+", true), {}, 1n], [P.unaryop("-", 1.5), {}, -1.5], [P.unaryop("+", 1.5), {}, 1.5],
    [P.call("abs", true), {}, 1n], [P.call("abs", -1.5), {}, 1.5], [P.call("bool", ""), {}, false], [P.call("float", 1n), {}, 1.0],
    [P.call("float", " 1e3 "), {}, 1000.0], [P.call("float", "inf"), {}, Infinity], [P.call("float", "-inf"), {}, -Infinity],
    [P.call("str", P.call("float", "nan")), {}, "nan"], [P.call("int", " 42 "), {}, 42n], [P.call("int", 2.7), {}, 2n],
    [P.call("int", -2.7), {}, -2n], [P.call("int", true), {}, 1n], [P.call("len", bytes(97, 98)), {}, 2n],
    [P.call("len", n("v")), { v: [1n, 2n] }, 2n], [P.call("len", n("v")), { v: { a: 1n } }, 1n], [P.call("len", "héllo"), {}, 5n],
    [P.call("max", 1n, 2.5), {}, 2.5], [P.call("min", 3n, 1n, 2n), {}, 1n], [P.call("round", 2.5), {}, 2n],
    [P.call("round", 3.5), {}, 4n], [P.call("round", -0.5), {}, 0n], [P.call("round", 7n), {}, 7n], [P.call("str", 1.0), {}, "1.0"],
    [P.call("str", bytes(120)), {}, "b'x'"], [P.call("str", n("v")), { v: null }, "None"], [P.call("str", true), {}, "True"],
    [P.call("str", 7n), {}, "7"], [P.call("str", "s"), {}, "s"], [P.import_("math", P.call("math.floor", 2.5)), {}, 2n],
    [P.unaryop("-", false), {}, 0n], [P.unaryop("not", n("v")), { v: ann }, false], [P.compare("==", 1.5, 1.5), {}, true],
    [P.binop("/", 1.5, 2n), {}, 0.75], [P.binop("&", 12n, 10n), {}, 8n], [P.binop("|", 12n, true), {}, 13n],
    [P.binop("^", true, true), {}, false], [P.binop("&", true, false), {}, false], [P.binop("|", false, true), {}, true],
    [P.binop("<<", true, 3n), {}, 8n], [P.binop(">>", -7n, 1n), {}, -4n], [P.unaryop("~", 5n), {}, -6n],
    [P.binop("|", P.binop("**", 2n, 70n), 1n), {}, 2n ** 70n + 1n],
    [P.compare("<", 1n, 2n, ["<", 3n]), {}, true], [P.compare("<", 1n, 3n, ["<", 2n]), {}, false], // a chain, link by link
    [P.compare("==", 1n, 1.0, ["in", n("v")]), { v: [1.0] }, true], [P.compare(">", 1n, 2n, ["<", "a"]), {}, false], // the first false link decides, its later operands not compared
    [P.boolop("and", 1n, 2n, 3n), {}, 3n], [P.boolop("and", 1n, 0n, n("missing")), {}, 0n], [P.boolop("or", 0n, "", 5n), {}, 5n],
    [P.boolop("or", 0n, ""), {}, ""], // n values: the first that decides, else the last
    [P.call("sum", P.generatorexp(P.binop("*", n("p"), n("q")), P.comprehension("p", n("v")),
      P.comprehension("q", n("v"), [P.compare(">", n("q"), n("p"))]))), { v: [1n, 2n, 3n] }, 11n],
    [P.call("all", P.generatorexp(P.compare(">", n("p"), 0n), P.comprehension("p", n("v")))), { v: [1n, 2n] }, true],
    [P.call("all", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [] }, true], [P.call("any", n("v")), { v: [0n, 3n] }, true],
    [P.call("any", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [0n, ""] }, false],
    [P.call("sum", P.generatorexp(1n, P.comprehension("p", n("v"), [P.compare(">", n("p"), 0n)]))), { v: [1n, -1n, 2n] }, 2n],
    [P.call("sum", P.generatorexp(1n, P.comprehension("c", "abc", [P.compare("!=", n("c"), "b"), P.compare("!=", n("c"), "c")]))), {}, 1n],
    [P.call("sum", n("v")), { v: Array(10).fill(0.1) }, 1.0], [P.call("sum", n("v")), { v: [1n, 0.1, 0.2, 0.3] }, 1.6],
    [P.call("sum", n("v")), { v: [1e+16, 1n, 1n] }, 1.0000000000000002e+16],
    [P.call("sum", n("v"), true), { v: Array(10).fill(0.1) }, 2.000000000000001], [P.call("sum", n("v")), { v: [true, true] }, 2n],
    [P.call("sum", n("v")), { v: [] }, 0n], [P.call("sum", n("v"), 1.5), { v: [] }, 1.5],
    [P.call("sum", n("v")), { v: [2n ** 62n, 2n ** 62n, 0.5] }, 9.223372036854776e+18],
    [P.call("sum", n("v")), { v: [(2n ** 63n) - 1n, 1n] }, 2n ** 63n],
    [P.call("sum", n("v")), { v: [-654.25, (-(2n ** 63n)) - 1n, -9n, (2n ** 63n) + 1n] }, -663.25],
    [P.call("sum", n("v"), 2n ** 64n), { v: [1n, 0.5] }, 1.8446744073709552e+19], [P.call("min", n("v")), { v: [3n, 1n, 2n] }, 1n],
    [P.call("max", P.generatorexp(P.binop("*", n("p"), 2n), P.comprehension("p", n("v")))), { v: [1n, 2.5] }, 5.0], [P.call("max", "abc"), {}, "c"],
    [P.call("min", P.call("set", n("v"))), { v: [2n, 1n] }, 1n], [P.call("len", P.call("set", n("v"))), { v: [1n, 1.0, true, 2n] }, 2n],
    [P.call("len", P.call("set")), {}, 0n], [P.unaryop("not", P.call("set")), {}, true],
    [P.call("all", n("v")), { v: [1n, 0n] }, false], [P.ifexp(0n, 1n, 2n), {}, 2n], [P.binop("-", 7n, 10n), {}, -3n],
    [P.compare("in", 2n, n("v")), { v: [1n, 2n] }, true], [P.compare("in", "b", "abc"), {}, true],
    [P.compare("in", "k", n("v")), { v: new Map([["k", 1n]]) }, true], [P.compare("not in", 3n, n("v")), { v: [1n, 2n] }, true],
    [P.compare("in", 1.0, n("v")), { v: [1n] }, true], [P.compare("in", 98n, new Uint8Array([97, 98])), {}, true],
    [P.compare("in", new Uint8Array([98]), new Uint8Array([97, 98])), {}, true],
    [P.compare("in", new Uint8Array([98, 97]), new Uint8Array([97, 98])), {}, false],
    [P.compare("in", n("w"), n("v")), { v: [[1n]], w: [1n] }, true], [P.compare("in", 1n, P.call("set", n("v"))), { v: [1n] }, true],
    [P.compare("in", 3n, P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [1n, 2n] }, false], [P.subscript(n("v"), 1n), { v: [10n, 20n] }, 20n],
    [P.subscript(n("v"), -1n), { v: [10n, 20n] }, 20n], [P.subscript("abc", 1n), {}, "b"], [P.subscript(new Uint8Array([97, 98]), 0n), {}, 97n],
    [P.subscript(n("v"), true), { v: [10n, 20n] }, 20n], [P.subscript(n("v"), 1.0), { v: new Map([[1n, "x"]]) }, "x"],
    [P.subscript(n("v"), "k"), { v: new Map([["k", 1n]]) }, 1n], [P.call("all", P.subscript(n("v"), 0n)), { v: [[1n, 1n]] }, true],
    [P.call("sum", bytes(1, 2)), {}, 3n], [P.call("max", bytes(97, 98)), {}, 98n],
    [P.call("len", P.call("set", n("v"))), { v: { a: 1n, b: 2n } }, 2n], [P.subscript(n("v"), "b"), { v: { a: 1n, b: 2n } }, 2n],
    [P.call("sum", P.attribute(n("v"), "ranks")), { v: squad }, 6n], [P.subscript(P.attribute(n("v"), "ranks"), -1n), { v: squad }, 3n],
  ];
  const pyEqual = (a: unknown, b: unknown) => Repr.typeName(a) === Repr.typeName(b)
    && (a instanceof Uint8Array ? F.sameNative(a, b) : a === b);
  for (const [expression, variables, expected] of VALUES) {
    const value = PV.OfAny(py(expression), new PV.Scope(variables, { modules }));
    assert(pyEqual(value, expected), `${PText.ToText(py(expression))}: ${Repr.repr(value)}, not ${Repr.repr(expected)}`);
  }
  const ERRORS: [unknown, Record<string, unknown>, Function, string][] = [ // (expression, variables, error, message)
    [P.compare("<", "a", 1n), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"],
    [P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"],
    [P.binop("-", "a", 1n), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"],
    [P.binop("&", 1.5, 1n), {}, TypeError, "unsupported operand type(s) for &: 'float' and 'int'"],
    [P.binop("<<", 1n, -1n), {}, ValueError, "negative shift count"], [P.binop(">>", 1n, -1n), {}, ValueError, "negative shift count"],
    [P.unaryop("~", 1.5), {}, TypeError, "bad operand type for unary ~: 'float'"],
    [P.binop("/", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("//", 1.5, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("%", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("**", 0.0, -1n), {}, ZeroDivisionError, "zero to a negative power"],
    [P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"],
    [P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"],
    [P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"],
    [P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"],
    [P.call("int", NaN), {}, ValueError, "cannot convert float NaN to integer"],
    [P.call("int", Infinity), {}, OverflowError, "cannot convert float infinity to integer"],
    [P.call("len", 1n), {}, TypeError, "object of type 'int' has no len()"],
    [P.call("max", 1n, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"],
    [P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"],
    [P.call("float", n("v")), { v: null }, TypeError, "float() argument must be a string or a real number, not 'NoneType'"],
    [P.call("int", n("v")), { v: null }, TypeError,
      "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"],
    [P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"],
    [P.subscript(n("t"), "y"), { t: { x: 1n } }, KeyError, "'y'"],
    [P.subscript(n("t"), "x"), { t: 3n }, TypeError, "'int' object is not subscriptable"],
    [P.call(P.subscript(n("t"), "f")), { t: { f: [() => 1n][0] } }, TypeError, "calling '<lambda>' is not allowed by the scope"],
    [P.call("len", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [1n] }, TypeError, "object of type 'generator' has no len()"],
    [P.call("sum", n("v")), { v: ["a"] }, TypeError, "unsupported operand type(s) for +: 'int' and 'str'"],
    [P.call("sum", n("v")), { v: [0.5, "a"] }, TypeError, "unsupported operand type(s) for +: 'float' and 'str'"],
    [P.call("sum", n("v"), "a"), { v: [1n] }, TypeError, "sum() can't sum strings [use ''.join(seq) instead]"],
    [P.call("sum", n("v"), new Uint8Array([])), { v: [1n] }, TypeError, "sum() can't sum bytes [use b''.join(seq) instead]"],
    [P.call("sum", n("v")), { v: [0.5, 10n ** 400n] }, OverflowError, "int too large to convert to float"],
    [P.call("min", n("v")), { v: [] }, ValueError, "min() iterable argument is empty"],
    [P.call("max", P.generatorexp(n("p"), P.comprehension("p", n("v")))), { v: [] }, ValueError, "max() iterable argument is empty"],
    [P.call("max"), {}, TypeError, "max expected at least 1 argument, got 0"],
    [P.call("min", 1n), {}, TypeError, "'int' object is not iterable"],
    [P.call("set", n("v")), { v: [[1n]] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.call("all", 5n), {}, TypeError, "'int' object is not iterable"],
    [P.generatorexp(n("p"), P.comprehension("p", 5n)), {}, TypeError, "'int' object is not iterable"],
    [P.compare("in", 1n, 5n), {}, TypeError, "argument of type 'int' is not a container or iterable"],
    [P.compare("in", 1n, "abc"), {}, TypeError, "'in <string>' requires string as left operand, not int"],
    [P.compare("in", "a", new Uint8Array([97, 98])), {}, TypeError, "a bytes-like object is required, not 'str'"],
    [P.compare("in", 256n, new Uint8Array([97, 98])), {}, ValueError, "byte must be in range(0, 256)"],
    [P.compare("in", n("w"), n("v")), { v: new Map([[1n, 2n]]), w: [1n] }, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"],
    [P.compare("in", n("w"), P.call("set", n("v"))), { v: [1n], w: [1n] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.subscript(n("v"), 5n), { v: [1n] }, IndexError, "list index out of range"],
    [P.subscript(n("v"), -3n), { v: [1n, 2n] }, IndexError, "list index out of range"],
    [P.subscript("abc", 5n), {}, IndexError, "string index out of range"],
    [P.subscript(new Uint8Array([97]), 3n), {}, IndexError, "index out of range"],
    [P.subscript(n("v"), P.binop("**", 2n, 70n)), { v: [1n] }, IndexError, "cannot fit 'int' into an index-sized integer"],
    [P.subscript(n("v"), 1.5), { v: [1n] }, TypeError, "list indices must be integers or slices, not float"],
    [P.subscript("abc", 1.5), {}, TypeError, "string indices must be integers, not 'float'"],
    [P.subscript(new Uint8Array([97, 98]), 1.5), {}, TypeError, "byte indices must be integers or slices, not float"],
    [P.subscript(5n, 0n), {}, TypeError, "'int' object is not subscriptable"],
    [P.subscript(n("v"), 5n), { v: new Map([["k", 1n]]) }, KeyError, "5"],
    [P.subscript(n("v"), n("w")), { v: new Map([["k", 1n]]), w: ["k"] }, TypeError, "cannot use 'list' as a dict key (unhashable type: 'list')"],
    [P.binop("@@", 1n, 2n), {}, NotImplementedError, "'@@' is not a core operation"],
    [(u: any) => u[P.LITERAL]((r: any) => r), {}, ValueError, "a literal needs a value"],
    // The evaluator is shallow: what it does not evaluate it refuses by name, before evaluating anything
    [P.binop("@", 1n, 2n), {}, NotImplementedError, "Python3 binop '@' is not evaluated"],
    [P.compare("<", 1n, 2n, ["is", 3n]), {}, NotImplementedError, "Python3 compare 'is' is not evaluated"],
    [P.compare("<", 1n, 2n, ["===", 3n]), {}, NotImplementedError, "'===' is not a core operation"],
    [P.lambda_(1n), {}, NotImplementedError, "Python3 lambda is not evaluated"],
    [P.constant(null), {}, NotImplementedError, "Python3 constant is not evaluated"],
    [P.tuple_(1n, 2n), {}, NotImplementedError, "Python3 tuple is not evaluated"],
    [P.subscript(n("v"), P.slice_(1n)), { v: [1n] }, NotImplementedError, "Python3 slice is not evaluated"],
    [P.call("abs", P.keyword("x", 1n)), {}, NotImplementedError, "Python3 keyword is not evaluated"],
    [P.call("abs", P.starred(n("v"))), { v: [1n] }, NotImplementedError, "Python3 starred is not evaluated"],
    [P.dict_([null, n("v")]), { v: {} }, NotImplementedError, "Python3 dict's ** item is not evaluated"],
    [P.generatorexp(1n, P.comprehension("p", n("v"), [], true)), { v: [1n] }, NotImplementedError,
      "Python3 async comprehension is not evaluated"],
    [P.generatorexp(1n, P.comprehension(P.tuple_(n("p")), n("v"))), { v: [1n] }, NotImplementedError,
      "Python3 tuple target is not evaluated"],
    [(u: any) => u[P.GENERATOREXP]((r: any) => r.elt(P.constant(1n)).generators([(a: any) => a.as_object((o: any) => o.iter(n("v")))])),
      { v: [1n] }, NotImplementedError, "Python3 missing target is not evaluated"],
    [(u: any) => u[P.GENERATOREXP]((r: any) => r.elt(P.constant(1n))), {}, ValueError, "a generatorexp needs a for clause"],
    [(u: any) => u[P.IMPORT]((r: any) => r.module("m")), {}, ValueError, "an import needs a body"],
  ];
  for (const [expression, variables, error, message] of ERRORS) {
    raises(error, () => PV.OfAny(py(expression), new PV.Scope(variables, { modules })), message);
  }
  assert(PV.OfAny(py(P.importfrom("numpy.ma", "masked", n("masked"))), new PV.Scope({}, { modules })) === masked);
  // A mapping is a Map or a plain object.
  assert(ev(P.subscript(t, "x"), { t: new Map([["x", 1n]]) }) === 1n && ev(P.unaryop("not", t), { t: new Map() }) === true);
  raises(KeyError, () => ev(P.subscript(t, "y"), { t: new Map() }), "'y'");
  assert(ev(P.call("len", t), { t: new Map([["x", 1n]]) }) === 1n);
}

<a id="DIA-03"></a>
## DIA-03 · Matlab: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
const s = M.identifier("s");
const matlabRule = mat(M.binary("&&", M.binary(">=", M.field(s, "age"), 18n), M.call("isfield", s, "email")));
{
  assert(MText.ToText(matlabRule) === 's.age >= 18 && isfield(s, "email")' && roundTrips(matlabRule));
  assert(MText.ToText(mat(M.binary("-", 1n, M.binary("-", 2n, 3n)))) === "1 - (2 - 3)"
    && MText.ToText(mat(M.binary("-", M.binary("-", 1n, 2n), 3n))) === "1 - 2 - 3");
  assert(MText.ToText(mat(M.binary(".*", M.binary("+", 1n, 2n), M.unary("-", M.unary("~", s))))) === "(1 + 2) .* -~s");
  assert(MText.ToText(mat(M.unary("-", M.binary("+", 1n, 2n)))) === "-(1 + 2)" && MText.ToText(mat(M.unary("~", -1n))) === "~-1");
  assert(MText.ToText(mat(M.binary("||", M.binary("&&", true, false), M.binary("&&", 1n, 0n)))) === "true && false || 1 && 0");
  assert(same([NaN, Infinity, -Infinity, -2.5, 'say "hi"', 7n].map((v) => MText.ToText(mat(M.constant(v)))), [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]));
  assert(MText.ToText(mat(M.field(M.field(s, "a"), "b"))) === "s.a.b");
  assert(M.DIALECT.infer(mat(matlabRule), { s: MD.Struct }) === MD.Logical);
  assert(M.DIALECT.infer(mat(M.binary("+", "a", 1n))) === MD.String && M.DIALECT.infer(mat(M.binary("+", true, 1n))) === MD.Double);
  assert(MD.Double.contains(1n) && MD.Struct.contains({}) && !MD.Struct.contains("s") && MD.of(1n) === MD.Double);
  assert(MD.Struct.contains(ann) && !MD.Struct.contains([1n])); // a data object is a struct
  assert(same(M.DIALECT.validate(mat(M.binary("^", 1n, 2n)), [], true), ["'^' is not a core operation"]));
  assert(same(M.DIALECT.validate(mat((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(new Uint8Array([120])))))),
    ["a literal cannot hold a bytes"]));
  assert(same(M.DIALECT.validate(mat(M.binary("+", null, 1n))), ["a binary needs a left"]));
  assert(equal(F.form(mat(M.call("f", 1n))).attributes, map({ callable: "f" }))); // MATLAB's call is the framework's
  raises(TypeError, () => M.constant(new Uint8Array([120])), "a literal cannot hold a bytes");
  raises(TypeError, () => M.binary("+", new Uint8Array([120]), 1n), "expected a term, a writer or a native value, got b'x'");
  raises(Error, () => term((u: any) => u.let((r: any) => r.name("x")), M.DIALECT), "'let' is not a member of a union value"); // Matlab's union has no let
  const notMatlab = term(Patterns.Expressions.let_("x", 1n, 2n), Patterns.Expressions.DIALECT);
  assert(same(M.DIALECT.validate(notMatlab), ["a Patterns let is not a Matlab term"]));
  raises(ValueError, () => M.DIALECT.kind(notMatlab), "a Patterns let is not a Matlab term");
}

<a id="DIA-04"></a>
## DIA-04 · Matlab evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
{
  assert(MV.OfAny(mat(matlabRule), { s: ann }) === true && MV.OfAny(mat(matlabRule), { s: bob }) === false);
  assert(MV.OfAny(mat(matlabRule), { s: { age: 20n, email: "x" } }) === true); // a mapping is a struct
  assert(MV.OfAny(mat(M.field(s, "age")), { s: ann }) === 30 && typeof MV.OfAny(mat(18n)) === "number"); // numbers are doubles
  assert(MV.OfAny(mat(M.binary("==", 1n, true))) === true && MV.OfAny(mat(M.binary("+", true, 1n))) === 2);
  assert(MV.OfAny(mat(M.binary("<", "apple", "banana"))) === true && MV.OfAny(mat(M.binary("==", "5", 5n))) === true);
  assert(MV.OfAny(mat(M.binary("==", "true", true))) === true && MV.OfAny(mat(M.binary("~=", "1.5", 1.5))) === false);
  assert(MV.OfAny(mat(M.binary("+", "a", 1n))) === "a1" && MV.OfAny(mat(M.binary("+", "x", 0.125))) === "x0.125");
  assert(same([NaN, Infinity, -Infinity, 12345.6, 1.5e-7, 123456.7, true, false].map((v) => MV.OfAny(mat(M.binary("+", "x", v)))), [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"])); // numbers as MATLAB writes them
  assert(MV.OfAny(mat(matlabRule), { s: new Map<string, unknown>([["age", 20n], ["email", "x"]]) }) === true); // a Map is a struct too
  assert(same(["~=", "<=", ">", ">=", "-", ".*"].map((op) => MV.OfAny(mat(M.binary(op, 2n, 3n)))), [true, true, false, false, -1, 6]));
  assert(MV.OfAny(mat(M.binary("&&", false, M.identifier("missing")))) === false); // short-circuits
  assert(MV.OfAny(mat(M.binary("||", 2n, M.identifier("missing")))) === true && MV.OfAny(mat(M.binary("||", 0n, 0n))) === false);
  assert(MV.OfAny(mat(M.unary("~", 0n))) === true && MV.OfAny(mat(M.unary("-", true))) === -1);
  assert(MV.OfAny(mat(M.call("isfield", 3n, "x"))) === false && MV.OfAny(mat(M.call("isfield", s, 1n)), { s: ann }) === false);
  raises(KeyError, () => MV.OfAny(mat(M.field(s, "email")), { s: bob }), 'Unrecognized field name "email".');
  raises(TypeError, () => MV.OfAny(mat(M.field(s, "age")), { s: 3n }), "Dot indexing is not supported for variables of this type.");
  raises(NameError, () => MV.OfAny(mat(M.identifier("missing"))), "Unrecognized function or variable 'missing'.");
  raises(TypeError, () => MV.OfAny(mat(M.binary("&&", "yes", true))), "Operands to the && operator must be convertible to logical scalar values.");
  raises(ValueError, () => MV.OfAny(mat(M.unary("~", NaN))), "NaN's cannot be converted to logicals.");
  raises(TypeError, () => MV.OfAny(mat(M.binary("-", "a", 1n))), "Operator '-' is not supported for operands of type 'string' and 'double'.");
  raises(TypeError, () => MV.OfAny(mat(M.binary(".*", s, true)), { s: ann }), "Operator '.*' is not supported for operands of type 'struct' and 'logical'.");
  raises(TypeError, () => MV.OfAny(mat(M.unary("-", "a"))), "Operator '-' is not supported for operands of type 'string'.");
  const adult = store.transaction(() => store.Expressions.Framework.Expression( // an expression, its root written by a writer
    (e: any) => e.root(M.binary(">", M.field(s, "age"), 18n))));
  assert(MText.ToText(adult) === "s.age > 18" && MV.OfAny(adult, { s: ann }) === true);
}

<a id="DIA-05"></a>
## DIA-05 · Matlab resolves functions from imports, the path and packages that the scope provides

In [ ]:
{
  const geo = { dist: (a: number, b: number) => (a * a + b * b) ** 0.5, area: (r: number) => 3 * r * r };
  const scope = new MV.Scope({ r: 2n }, { functions: { twice: (v: number) => 2 * v },
    packages: { geo, "geo.more": { area: () => 0n } } });
  const wild = mat(M.import_("geo.*", M.binary("+", M.call("dist", 3n, 4n), M.call("area", M.identifier("r")))));
  assert(MText.ToText(mat(wild)) === "import geo.*\ndist(3, 4) + area(r)" && M.DIALECT.validate(mat(wild), ["r"]).length === 0);
  assert(same(M.DIALECT.validate(mat(wild), ["r"], true), ["body: left: 'dist' is not a core operation",
    "body: right: 'area' is not a core operation"]));
  assert(equal(SY.imports(wild, M.DIALECT).map((i) => F.form(i).attributes), [map({ name: "geo.*" })]));
  assert(MV.OfAny(mat(wild), scope) === 17);
  assert(MV.OfAny(mat(M.import_("geo.dist", M.call("dist", 6n, 8n))), scope) === 10);
  assert(MV.OfAny(mat(M.call("geo.more.area", 1n)), scope) === 0 && MV.OfAny(mat(M.call("twice", 4n)), scope) === 8); // an int result is a double
  assert(MV.OfAny(mat(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5n)))), scope) === 0); // innermost wins
  raises(NameError, () => MV.OfAny(mat(M.call("dist", 3n, 4n)), scope), "Unrecognized function or variable 'dist'."); // not imported
  raises(ImportError, () => MV.OfAny(mat(M.import_("os.system", 1n)), scope), "Import argument 'os.system' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(mat(M.import_("geo.nope", 1n)), scope), "Import argument 'geo.nope' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(mat(M.import_("nope.*", 1n)), scope), "Import argument 'nope.*' cannot be found or cannot be imported.");
  raises(ValueError, () => MText.ToText(mat(M.binary("+", 1n, M.import_("geo.*", 2n)))), "an import can only enclose the whole expression");
  assert(same(M.DIALECT.validate(mat(M.import_("geo", 1n))), ["an import's name must be pkg.name or pkg.*, got 'geo'"]));
  assert(same(M.DIALECT.validate(mat(M.import_("geo.*.x", 1n))), ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]));
  assert(same(M.DIALECT.validate(mat(M.import_(7n as any, M.constant(1n)))), ["an import's name must be a str, got int"]));
}

<a id="DIA-06"></a>
## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
const r = X.name("r");
const excelRule = xl(X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18n),
  X.function("NOT", X.function("ISERROR", X.field(r, "email"))))));
{
  assert(XText.ToText(excelRule) === "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))" && roundTrips(excelRule));
  assert(XText.ToText(xl(X.infix("*", X.infix("+", 1n, 2n), X.prefix("-", X.infix("-", 3n, 4n))))) === "=(1 + 2) * -(3 - 4)");
  assert(XText.ToText(xl(X.infix("=", X.infix("-", 1n, X.infix("-", 2n, 3n)), -1n))) === "=1 - (2 - 3) = -1");
  assert(XText.ToText(xl(X.prefix("-", X.field(r, "first name")))) === "=-r.[first name]");
  assert(same([true, false, 'say "hi"', Infinity, 2.5, 3n].map((v) => XText.ToText(xl(X.constant(v)))), [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]));
  assert(X.DIALECT.validate(xl(excelRule), ["r"], true).length === 0
    && same(X.DIALECT.validate(xl(X.function("VLOOKUP", 1n, 2n)), [], true), ["'VLOOKUP' is not a core operation"]));
  assert(X.DIALECT.infer(xl(excelRule), { r: XD.Record }) === XD.Logical && X.DIALECT.infer(xl(X.infix("+", "1", true))) === XD.Number);
  assert(XD.of(1n) === XD.Number && XD.Errors.contains(XD.Error.of("#N/A")) && String(XD.Error.of("#N/A")) === "#N/A");
  assert(XD.Record.contains(ann) && !XD.Record.contains(1n));
  assert([true, 'say "hi"', 2.5, 3n, -0, NaN].every((v) => roundTrips(xl(X.constant(v)))));
  assert(same([...SY.free(excelRule, X.DIALECT)], ["r"]));
  raises(TypeError, () => X.constant(new Uint8Array([0])), "an Excel literal cannot hold a bytes");
  for (const [spec, problems] of [
    [(u: any) => u.literal((c: any) => c.value((v: any) => v.as_scalar(new Uint8Array([1])))), ["an Excel literal cannot hold a bytes"]],
    [(u: any) => u.cell((c: any) => c.sheet("S")), ["a cell needs an address"]],
    [(u: any) => u.call((c: any) => c.arguments([])), ["a call needs a callable"]],
    [(u: any) => u.infix((c: any) => c.operator("+").left(X.constant(1n))), ["an infix needs a right"]],
    [X.infix("+", null, 1n), ["an infix needs a left"]], // a writer leaves out a slot given null
    [(u: any) => u.literal((c: any) => c), ["a literal needs a value"]],
    [(u: any) => u.map((c: any) => c.name("p")), ["a map needs an array", "a map needs a body"]],
    [X.name("u"), ["variable 'u' is not bound"]],
    [X.function("NOT", 1n, 2n), ["NOT takes 1 arguments, got 2"]],
  ] as [unknown, string[]][]) {
    assert(same(X.DIALECT.validate(xl(spec)), problems), Repr.repr(X.DIALECT.validate(xl(spec))));
  }
  assert(same(X.DIALECT.validate(xl(X.infix("^", 1n, 2n)), [], true), ["'^' is not a core operation"]));
  const quantified = term(Patterns.Expressions.variable("xs").all("p", true), Patterns.Expressions.DIALECT); // a Patterns kind Excel does not read
  assert(same(X.DIALECT.validate(quantified), ["a Patterns quantifier is not an Excel term"]));
  raises(ValueError, () => XText.ToText(quantified), "a Patterns variable is not an Excel term"); // printing reads its terms first
  raises(ValueError, () => XText.ToText(xl(Patterns.Expressions.operation("SUM", Patterns.Expressions.variable("x"), 1n))),
    "a Patterns variable is not an Excel term"); // the same kinds, of another dialect
}

<a id="DIA-07"></a>
## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
{
  const Error = XV.Error;
  assert(XV.OfAny(xl(excelRule), { r: ann }) === true && XV.OfAny(xl(excelRule), { r: bob }) === false); // ISERROR finds the #FIELD!
  assert(XV.OfAny(xl(X.field(r, "email")), { r: bob }) === Error.of("#FIELD!") && XV.OfAny(xl(X.name("nope"))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.field(r, "age")), { r: { age: 5n } }) === 5 && XV.OfAny(xl(X.field(r, "x")), { r: 3n }) === Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.field(X.field(r, "a"), "b")), { r: {} }) === Error.of("#FIELD!")); // errors propagate
  assert(XV.OfAny(xl(X.infix("+", "1", true))) === 2 && XV.OfAny(xl(X.infix("*", "x", 1n))) === Error.of("#VALUE!")); // coercion
  assert(XV.OfAny(xl(X.infix("-", 1n, X.infix("+", X.name("u"), 1n)))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.prefix("-", "2"))) === -2 && XV.OfAny(xl(X.prefix("-", X.name("u")))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.infix(">", "a", 1n))) === true && XV.OfAny(xl(X.infix(">", true, "z"))) === true); // number < text < logical
  assert(XV.OfAny(xl(X.infix("=", "a", "A"))) === true && XV.OfAny(xl(X.infix("<>", 1n, "1"))) === true); // case; no coercion
  assert(same(["<", "<=", ">=", "-"].map((op) => XV.OfAny(xl(X.infix(op, 2n, 3n)))), [true, true, false, -1]));
  assert(XV.OfAny(xl(X.infix("=", 2n, 2n))) === true && XV.OfAny(xl(X.infix("+", false, 1n))) === 1);
  assert(XV.OfAny(xl(X.infix("+", "inf", 1n))) === Error.of("#VALUE!") && XV.OfAny(xl(X.infix("+", " 1e3 ", 1n))) === 1001); // decimal text
  assert(XV.OfAny(xl(X.infix("+", r, 1n)), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.field(r, "age")), { r: new Map([["age", 5n]]) }) === 5); // a Map is a record too
  assert(XV.OfAny(xl(X.infix("=", X.name("u"), 1n))) === Error.of("#NAME?") && XV.OfAny(xl(X.infix("=", r, 1n)), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.function("AND", 1n, true))) === true && XV.OfAny(xl(X.function("OR", 0n, false))) === false);
  assert(XV.OfAny(xl(X.function("AND", false, X.name("u")))) === Error.of("#NAME?")); // AND evaluates every argument
  assert(XV.OfAny(xl(X.function("OR", "yes", true))) === Error.of("#VALUE!") && XV.OfAny(xl(X.function("NOT", 0n))) === true);
  assert(XV.OfAny(xl(X.function("NOT", X.name("u")))) === Error.of("#NAME?"));
  assert(XV.OfAny(xl(X.function("IF", true, 1n, X.name("u")))) === 1 && XV.OfAny(xl(X.function("IF", 0n, 1n, "no"))) === "no");
  assert(XV.OfAny(xl(X.function("IF", X.name("u"), 1n, 2n))) === Error.of("#NAME?")); // IF evaluates only the branch it takes
  assert(XV.OfAny(xl(X.field(r, "age")), { r: ann }) === 30);
  assert(XV.OfAny(xl(X.function("ISERROR", 1n))) === false && XV.OfAny(xl(X.let_("x", 2n, X.infix("*", X.name("x"), 3n)))) === 6);
  const adult = store.transaction(() => store.Expressions.Framework.Expression( // an expression, its root written by a writer
    (e: any) => e.root(X.function("AND", X.infix(">=", X.field(r, "age"), 18n)))));
  assert(XV.OfAny(adult, { r: ann }) === true && XText.ToText(adult) === "=AND(r.age >= 18)");
}

<a id="DIA-08"></a>
## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
{
  const Error = XV.Error;
  const rates = new XV.Workbook({}, { Rates: { B2: 0.5 } }, { name: "Rates.xlsx" });
  const book = new XV.Workbook({ base: 10n }, { Sheet1: { a1: 2n, $A$2: 3n }, "My Sheet": { C3: "x" } },
    { books: { "Rates.xlsx": rates }, add_ins: { double: (v: number) => v * 2 } });
  const total = xl(X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base"))));
  assert(XText.ToText(xl(total)) === "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" && XV.OfAny(xl(total), book) === 21 && roundTrips(total));
  assert(X.DIALECT.validate(xl(total), ["base"]).length === 0
    && same(X.DIALECT.validate(xl(total), ["base"], true), ["right: 'DOUBLE' is not a core operation"]));
  assert(XText.ToText(xl(X.cell("C3", "My Sheet"))) === "='My Sheet'!C3" && XV.OfAny(xl(X.cell("C3", "My Sheet")), book) === "x");
  assert(XText.ToText(xl(X.cell("A1", "Bob's", "My Book.xlsx"))) === "='[My Book.xlsx]Bob''s'!A1");
  assert(XV.OfAny(xl(X.cell("$a$2")), book) === 3 && XV.OfAny(xl(X.cell("Z9")), book) === 0); // an empty cell is 0
  assert(XV.OfAny(xl(X.cell("A1", "Sheet1", "Book1")), book) === 2); // this book, named
  assert(XV.OfAny(xl(X.cell("A1", "Nope")), book) === Error.of("#REF!") && XV.OfAny(xl(X.cell("A1", "S", "Other.xlsx")), book) === Error.of("#REF!"));
  assert(XV.OfAny(xl(X.function("NOPE", 1n)), book) === Error.of("#NAME?")
    && XV.OfAny(xl(X.function("double", X.name("u"))), book) === Error.of("#NAME?")); // add-ins ignore case; errors in their arguments propagate
  assert(XV.OfAny(xl(X.cell("A1")), new XV.Workbook()) === Error.of("#REF!")); // a workbook has no sheet until given one
  assert(XV.OfAny(xl(X.name("x")), { x: 1n }) === 1 && new XV.Workbook().sheet === "Sheet1");
  raises(ValueError, () => new XV.Workbook({}, { Sheet1: { total: 1n } }), "not a cell address: 'total'");
  assert(same(X.DIALECT.validate(xl(X.cell("A0"))), ["a cell's address must be a column and a row, like A1, got 'A0'"]));
  assert(same(X.DIALECT.validate(xl(X.cell("A1", null, "B.xlsx"))), ["a cell in another book needs a sheet"]));
  assert(X.DIALECT.infer(xl(X.cell("A1"))) === D.Anything && X.address("$b$12") === "B12" && X.address("B") === null);
}

<a id="DIA-09"></a>
## DIA-09 · Latex: notation in math mode, rendered and checked, with no evaluator

In [ ]:
{
  const [L, LD] = [Latex.Expressions, Latex.Domains];
  const [a, person] = [L.symbol("a"), L.symbol("person")];
  const notation = tex(L.where("a", L.member(person, "age"), L.binary("\\land", L.binary("\\geq", a, 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", person, "email")), L.binary(">", L.frac(a, 2n), 1.5)))));
  assert(LText.ToText(tex(notation)) === "a \\geq 18 \\land (\\lnot \\operatorname{has}(\\mathit{person}, \\text{email}) \\lor \\frac{a}{2} > 1.5) "
    + "\\quad \\text{where } a = \\mathit{person}.\\mathit{age}" && roundTrips(notation));
  assert(LText.ToText(tex(L.binary("\\cdot", L.binary("+", 1n, 2n), L.unary("-", L.binary("-", 3n, -4n))))) === "(1 + 2) \\cdot -(3 - -4)");
  assert(LText.ToText(tex(L.binary("\\implies", L.binary("\\implies", 1n, 2n), L.binary("\\implies", 3n, 4n))))
    === "(1 \\implies 2) \\implies 3 \\implies 4"); // right-associative
  assert(LText.ToText(tex(L.binary("<", L.binary("<", 1n, 2n), 3n))) === "(1 < 2) < 3"); // comparisons do not chain
  assert(LText.ToText(tex(L.member(L.binary("+", person, 1n), "x_y"))) === "(\\mathit{person} + 1).\\mathit{x\\_y}");
  assert(LText.ToText(tex(L.where("b", 1n, L.where("a", 2n, a)))) === "(a \\quad \\text{where } a = 2) \\quad \\text{where } b = 1");
  assert(same([NaN, Infinity, -Infinity, 1e-7, 1e16, -2n, true, false, "a_b {c}"].map((v) => LText.ToText(tex(L.constant(v)))), [
    "\\mathrm{NaN}", "\\infty", "-\\infty", "1 \\times 10^{-7}", "1 \\times 10^{16}", "-2", "\\mathrm{true}",
    "\\mathrm{false}", "\\text{a\\_b \\{c\\}}"]));
  assert(LText.ToText(tex(L.constant("\\$&#%~^"))) === "\\text{\\textbackslash{}\\$\\&\\#\\%\\textasciitilde{}\\textasciicircum{}}");
  assert(LText.ToText(tex(L.binary("\\cdot", L.constant(1e16), 2n))) === "1 \\times 10^{16} \\cdot 2");
  assert(L.DIALECT.validate(tex(notation), ["person"], true).length === 0
    && same(L.DIALECT.validate(tex(L.binary("\\times", 1n, 2n)), [], true), ["'\\\\times' is not a core operation"]));
  assert(same(L.DIALECT.validate(notation), ["value: value: variable 'person' is not bound",
    "body: right: left: operand: argument 0: variable 'person' is not bound"]));
  assert(same(L.DIALECT.validate(tex((u: any) => u.binary((r: any) => r.operator("+").left(L.DIALECT.spec(1n))))), ["a binary needs a right"]));
  assert(same(L.DIALECT.validate(tex((u: any) => u.variable((r: any) => r))), ["a variable needs a name"]));
  assert(same(L.DIALECT.validate(tex(L.constant(new Uint8Array([0])))), ["a Latex literal cannot hold a bytes"])); // the framework's literal, Latex's check
  assert(L.DIALECT.validate(tex(L.function("max", 1n, 2n))).length === 0 && same(L.DIALECT.validate(tex(L.function("has", 1n))), ["has takes 2 arguments, got 1"]));
  assert(L.DIALECT.infer(tex(notation), { person: D.Anything }) === LD.Truth && L.DIALECT.infer(tex(L.frac(1n, 2.5))) === LD.Number);
  raises(TypeError, () => L.DIALECT.infer(tex(L.binary("\\land", 1n, true))), "\\land cannot take (number, truth)");
  raises(TypeError, () => L.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'");
  assert(F.same(F.form(tex(L.frac(notation, 1n))).arguments[0], notation)); // a term written into another is copied
  assert(LD.of("x") === LD.Text && !("Evaluators" in Latex)); // notation is not evaluated
}

<a id="DIA-10"></a>
## DIA-10 · Bitwise operations: Python's operators, and MATLAB's and Excel's bit functions

In [ ]:
{
  // Python: rendered with Python's precedence, and evaluated by its rules (DIA-02's tables have more).
  const [a, b, c, d, x] = ["a", "b", "c", "d", "x"].map((name) => P.name(name)) as [any, any, any, any, any];
  const sources: [unknown, string][] = [
    [P.binop("|", a, P.binop("^", b, P.binop("&", c, P.binop(">>", P.binop("<<", d, 1n), 2n)))), "a | b ^ c & d << 1 >> 2"],
    [P.binop("&", P.binop("|", a, b), c), "(a | b) & c"], [P.binop("+", P.unaryop("~", x), 1n), "~x + 1"],
    [P.binop("+", P.binop("<<", a, b), c), "(a << b) + c"], [P.compare("==", P.binop("&", x, 1n), 1n), "x & 1 == 1"],
    [P.unaryop("-", P.unaryop("~", x)), "-~x"],
  ];
  for (const [expression, source] of sources) assert(PText.ToText(py(expression)) === source, source);
  const mixed = py(P.binop("^", P.binop("|", P.binop("&", x, 12n), P.binop(">>", x, 2n)), P.binop("<<", x, 1n)));
  assert(PText.ToText(mixed) === "(x & 12 | x >> 2) ^ x << 1" && PV.OfAny(mixed, { x: 13n }) === 21n);
  assert(P.DIALECT.infer(py(P.binop("&", true, false))) === PD.Bool && P.DIALECT.infer(py(P.binop("<<", 1n, true))) === PD.Int);
  assert(P.DIALECT.infer(py(P.unaryop("~", x)), { x: PD.Bool }) === PD.Int);
  // MATLAB: the bit functions take doubles holding integers of 53 bits.
  assert(MV.OfAny(mat(M.call("bitand", 12n, 10n))) === 8 && MV.OfAny(mat(M.call("bitor", 12n, true))) === 13);
  assert(MV.OfAny(mat(M.call("bitxor", 12n, 10n))) === 6 && MV.OfAny(mat(M.call("bitshift", 1n, 3n))) === 8);
  assert(MV.OfAny(mat(M.call("bitshift", 16n, M.unary("-", 2n)))) === 4 && MV.OfAny(mat(M.call("bitshift", 2 ** 52, 1n))) === 0); // beyond 53 bits
  assert(MText.ToText(mat(M.call("bitshift", M.identifier("x"), M.unary("-", 2n)))) === "bitshift(x, -2)");
  assert(M.DIALECT.infer(mat(M.call("bitand", 1n, true))) === MD.Double);
  for (const call of [M.call("bitand", -1n, 1n), M.call("bitand", 1.5, 1n), M.call("bitor", 2 ** 54, 1n), M.call("bitshift", 1n, 0.5)]) {
    raises(ValueError, () => MV.OfAny(mat(call)), "Double inputs must have integer values in the range of ASSUMEDTYPE.");
  }
  raises(TypeError, () => MV.OfAny(mat(M.call("bitor", "a", 1n))), "Undefined function 'bitor' for input arguments of type 'string'.");
  // Excel: the bit functions take integers below 2^48, converted as arithmetic converts; otherwise #NUM!.
  assert(XV.OfAny(xl(X.function("BITAND", 12n, 10n))) === 8 && XV.OfAny(xl(X.function("BITOR", "12", true))) === 13);
  assert(XV.OfAny(xl(X.function("BITXOR", 12n, 10n))) === 6 && XV.OfAny(xl(X.function("BITLSHIFT", 1n, 3n))) === 8);
  assert(XV.OfAny(xl(X.function("BITRSHIFT", 16n, 2n))) === 4 && XV.OfAny(xl(X.function("BITRSHIFT", 1n, -3n))) === 8);
  assert(XV.OfAny(xl(X.function("BITLSHIFT", 16n, -2n))) === 4 && XText.ToText(xl(X.function("BITXOR", X.name("x"), 1n))) === "=BITXOR(x, 1)");
  for (const formula of [X.function("BITAND", -1n, 1n), X.function("BITAND", 1.5, 1n), X.function("BITOR", 2n ** 48n, 1n),
    X.function("BITLSHIFT", 1n, 54n), X.function("BITLSHIFT", 1n, 48n), X.function("BITRSHIFT", 1n, 0.5)]) {
    assert(XV.OfAny(xl(formula)) === XV.Error.of("#NUM!"), XText.ToText(xl(formula)));
  }
  assert(XV.OfAny(xl(X.function("BITAND", "a", 1n))) === XV.Error.of("#VALUE!"));
  assert(XV.OfAny(xl(X.function("BITXOR", 1n, X.name("nope")))) === XV.Error.of("#NAME?"));
  assert(X.DIALECT.infer(xl(X.function("BITAND", 1n, 2n))) === XD.Number);
}

<a id="DIA-11"></a>
## DIA-11 · C: C and C++ expressions over their arithmetic types, with C's precedence, typed constants and casts

In [ ]:
{
  const [x, y, device] = [C.identifier("x"), C.identifier("y"), C.identifier("device")];
  const cases: [unknown, string][] = [
    [C.binary("+", x, C.binary("*", y, 2n)), "x + y * 2"], [C.binary("*", C.binary("+", x, y), 2n), "(x + y) * 2"],
    [C.binary("-", x, C.binary("-", y, 1n)), "x - (y - 1)"], [C.binary("<<", C.binary("+", x, 1n), 2n), "x + 1 << 2"],
    [C.binary("&", C.binary("==", x, 1n), y), "x == 1 & y"], [C.binary("==", C.binary("&", x, 1n), y), "(x & 1) == y"],
    [C.binary("||", C.binary("&&", x, y), C.unary("!", x)), "x && y || !x"],
    [C.binary("|", C.binary("^", x, y), C.binary("%", x, y)), "x ^ y | x % y"],
    [C.binary("<", C.binary(">>", x, 1n), C.binary("/", y, 2n)), "x >> 1 < y / 2"],
    [C.conditional(x, y, C.conditional(y, 1n, 2n)), "x ? y : y ? 1 : 2"], [C.conditional(C.conditional(x, y, 1n), 2n, 3n), "(x ? y : 1) ? 2 : 3"],
    [C.unary("-", C.unary("-", x)), "- -x"], [C.unary("-", -1n), "- -1"], [C.unary("~", C.unary("!", x)), "~!x"],
    [C.cast("uint8_t", C.binary("+", x, 1n)), "(uint8_t)(x + 1)"], [C.unary("-", C.cast("int", x)), "-(int)x"],
    [C.member(C.member(device, "a"), "b", "->"), "device.a->b"], [C.subscript(C.member(device, "ports"), C.binary("+", x, 1n)), "device.ports[x + 1]"],
    [C.call("f", x, C.binary("+", y, 1n)), "f(x, y + 1)"], [C.member(C.binary("+", x, y), "a"), "(x + y).a"],
    [C.constant(5n, "unsigned int"), "5u"], [C.constant(5n, "unsigned long"), "5ul"], [C.constant(5n, "long long"), "5ll"],
    [C.constant(5n, "unsigned long long"), "5ull"], [C.constant(5n, "long"), "5l"], [C.constant(5n, "int"), "5"],
    [C.constant(1.5, "float"), "1.5f"], [C.constant("1.5", "long double"), "1.5L"], [C.constant("1", "long double"), "1.0L"],
    [C.constant("Infinity", "long double"), "INFINITY"], [C.constant(5n, "uint8_t"), "(uint8_t)5"], [C.constant(2.5, "double"), "2.5"],
    [C.constant(1.0), "1.0"], [C.constant(1e300), "1e+300"], [C.constant(NaN), "NAN"], [C.constant(-Infinity), "-INFINITY"], [C.constant(Infinity), "INFINITY"],
    [C.constant(true), "true"], [C.constant(false), "false"], [C.constant('a"\\\n\t\r'), '"a\\"\\\\\\n\\t\\r"'],
    [C.constant(-5n), "-5"], [C.binary("-", x, -5n), "x - -5"],
  ];
  for (const [expression, source] of cases) assert(CText.ToText(cc(expression)) === source, `${source}: ${CText.ToText(cc(expression))}`);
  const sample = cc(C.conditional(C.binary("&&", C.unary("!", C.member(device, "a")), C.binary(
    ">", C.subscript(C.member(device, "b", "->"), 0n), C.constant(1.5, "float"))), C.cast("uint8_t", C.call("f", 1n)), C.constant(5n, "unsigned int")));
  assert(roundTrips(sample) && CText.ToText(sample) === "!device.a && device->b[0] > 1.5f ? (uint8_t)f(1) : 5u");
  // Validation: types are C's arithmetic types, and hold the constant's value.
  assert(same(C.DIALECT.validate(cc(C.constant(5n, "uint9_t"))), ["a constant's type must be one of C's arithmetic types, got 'uint9_t'"]));
  assert(same(C.DIALECT.validate(cc(C.constant(300n, "uint8_t"))), ["a constant of uint8_t cannot hold 300"]));
  assert(same(C.DIALECT.validate(cc(C.cast("word", x)), ["x"]), ["a cast's type must be one of C's arithmetic types, got 'word'"]));
  assert(same(C.DIALECT.validate(cc(C.member(x, "a", "=>")), ["x"]), ["a member's operator must be '.' or '->', got '=>'"]));
  assert(same(C.DIALECT.validate(cc(C.binary("**", x, 1n)), ["x"], true), ["'**' is not a core operation"]));
  assert(C.DIALECT.validate(cc(C.call("f", x)), ["x"], true).length === 0); // functions are the scope's
  assert(same(C.DIALECT.validate(cc((u: any) => u.cast((r: any) => r))), ["a cast needs a type", "a cast needs an operand"]));
  assert(same(C.DIALECT.validate(cc(C.constant(null as any))), ["a constant needs a value"]));
  raises(ValueError, () => CText.ToText(cc(C.constant(null as any))), "a constant needs a value"); // an invalid term has no source
  raises(TypeError, () => C.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'"); // no C constant holds bytes
  assert(same(C.DIALECT.validate(cc(C.binary("+", x, 1n))), ["left: variable 'x' is not bound"]));
  // Types under LP64, the integer promotions and the usual arithmetic conversions.
  const T = (name: string) => CD.TYPES.get(name) as CDialect.Domains.CType;
  assert(T("int32_t").equals(T("int")) && T("int32_t").name() === "int32_t" && String(T("size_t")) === "size_t" && T("size_t").equals(T("unsigned long")));
  assert(CD.promoted(T("uint8_t")) === CD.Int && CD.promoted(CD.Bool) === CD.Int && CD.promoted(T("long")) === T("long"));
  assert(CD.common(CD.Int, T("unsigned int")).name() === "unsigned int" && CD.common(T("long"), T("unsigned int")).name() === "long");
  assert(CD.common(T("float"), T("long")).name() === "float" && CD.common(CD.Double, T("long double")).name() === "long double");
  assert(CD.common(T("unsigned long"), T("long long")).equals(T("unsigned long")) && CD.common(T("short"), T("char")) === CD.Int);
  assert(CD.common(CD.Int, T("long long")).name() === "long long" && CD.common(T("long"), CD.Int).name() === "long");
  const ofs = [1n, 2n ** 40n, 2n ** 63n, true, 1.5, "a"].map((v) => CD.of(v));
  assert(same(ofs, [CD.Int, T("long"), T("unsigned long"), CD.Bool, CD.Double, CD.String]));
  assert(CD.of(2n ** 64n) === D.Anything && CD.of(null) === D.Anything);
  assert(T("uint8_t").contains(255n) && !T("uint8_t").contains(256n) && T("int8_t").contains(-128n) && !T("int8_t").contains(-129n));
  assert(T("float").contains(1.5) && T("long double").contains("1.5") && CD.Bool.contains(true) && !CD.Int.contains(true));
  assert(!CD.Int.contains(1.5) && CD.Int.includes(T("int32_t")) && CD.Int.includes(new D.OfUnion(CD.Int)) && !CD.Int.includes(T("long")));
  const infer = (expression: unknown, environment?: Record<string, D.Domain>) => C.DIALECT.infer(cc(expression), environment);
  assert(infer(C.binary("+", C.constant(1n, "uint8_t"), 1n)) === CD.Int && (infer(C.binary("+", 1n, C.constant(1n, "unsigned int"))) as any).name() === "unsigned int");
  assert(infer(C.binary("<<", C.constant(1n, "uint8_t"), C.constant(1n, "long"))) === CD.Int && infer(C.binary("<", 1n, 2.0)) === CD.Bool);
  assert(infer(C.binary("&&", x, 1n), { x: D.Anything }) === CD.Bool && infer(C.binary("+", x, 1n), { x: D.Anything }) === D.Anything);
  assert(T("uint16_t").equals(infer(C.cast("uint16_t", x), { x: D.Anything })) && T("uint8_t").equals(infer(C.constant(5n, "uint8_t"))));
  assert(infer(C.unary("~", C.constant(1n, "uint8_t"))) === CD.Int && infer(C.unary("-", 1.5)) === CD.Double && infer(C.unary("!", 1.5)) === CD.Bool);
  assert((infer(C.binary("%", C.constant(7n, "uint32_t"), C.constant(2n, "uint32_t"))) as any).name() === "unsigned int");
  assert(infer(C.conditional(x, 1n, 2n), { x: D.Anything }) === D.Anything && infer(C.member(x, "a"), { x: D.Anything }) === D.Anything);
  const refused: [unknown, string][] = [[C.binary("%", 1.5, 1n), "% cannot take (double, int); it takes (integer, integer) -> their common type"],
    [C.binary("+", "a", 1n), "+ cannot take (string, int)"], [C.binary("<<", 1.5, 1n), "<< cannot take (double, int)"],
    [C.unary("~", 1.5), "~ cannot take (double); it takes (integer) -> its promoted type"]];
  for (const [expression, message] of refused) raises(TypeError, () => infer(expression), message);
}

<a id="DIA-12"></a>
## DIA-12 · C evaluates by C's rules: promotions and conversions, wrapping unsigned arithmetic, and undefined behavior raising

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => CV.OfAny(cc(expression), variables) as any;
  const x = C.identifier("x");
  const U = (value: bigint) => C.constant(value, "unsigned int");
  assert(ev(C.binary("+", U(4294967295n), 1n)) === 0n && ev(C.binary("-", U(0n), 1n)) === 4294967295n && ev(C.binary("*", U(65536n), U(65536n))) === 0n);
  assert(ev(C.binary("<", -1n, U(0n))) === false && ev(C.binary("<", -1n, C.constant(0n, "unsigned short"))) === true); // promoted to int
  assert(ev(C.binary("+", C.constant(200n, "uint8_t"), C.constant(100n, "uint8_t"))) === 300n); // in int
  assert(ev(C.binary("/", -7n, 2n)) === -3n && ev(C.binary("%", -7n, 2n)) === -1n && ev(C.binary("/", 7n, -2n)) === -3n && ev(C.binary("%", 7n, -2n)) === 1n);
  assert(ev(C.binary("/", 7.0, 2n)) === 3.5 && ev(C.binary("/", 1.0, 0.0)) === Infinity && Number.isNaN(ev(C.binary("/", 0.0, 0.0))));
  assert(ev(C.binary("+", C.cast("float", 1.5), C.cast("float", 0.1))) === 1.600000023841858 && ev(C.binary("-", 1.5, 1n)) === 0.5);
  assert(ev(C.binary("*", C.cast("long double", 1.5), 2n)) === "3");
  assert(ev(C.binary("&", 12n, 10n)) === 8n && ev(C.binary("^", 12n, 10n)) === 6n && ev(C.binary("|", -1n, U(0n))) === 4294967295n);
  assert(ev(C.binary("<<", U(1n), 31n)) === 2147483648n && ev(C.binary(">>", -8n, 1n)) === -4n && ev(C.binary("<<", C.constant(1n, "uint8_t"), 8n)) === 256n);
  assert(ev(C.unary("-", U(1n))) === 4294967295n && ev(C.unary("~", C.constant(0n, "uint8_t"))) === -1n && ev(C.unary("~", U(0n))) === 4294967295n);
  assert(ev(C.unary("+", true)) === 1n && ev(C.unary("-", 1.5)) === -1.5 && ev(C.unary("+", 1.5)) === 1.5);
  assert(ev(C.unary("!", 0.0)) === true && ev(C.unary("!", -0)) === true && ev(C.unary("!", NaN)) === false && ev(C.unary("!", 3n)) === false);
  assert(ev(C.binary("==", NaN, NaN)) === false && ev(C.binary("!=", NaN, 1.0)) === true && ev(C.binary("==", -0, 0.0)) === true);
  assert(ev(C.binary("<", 1n, 1.5)) === true && ev(C.binary(">=", C.cast("long double", 2n), 1n)) === true && ev(C.binary(">", 2n, 1n)) === true);
  assert(ev(C.binary("<=", Infinity, Infinity)) === true && ev(C.binary("<", C.cast("long double", NaN), 1n)) === false);
  assert(ev(C.binary("&&", 0n, C.binary("/", 1n, 0n))) === false && ev(C.binary("||", 2n, C.binary("/", 1n, 0n))) === true); // decided early
  assert(ev(C.binary("&&", 1n, 2.5)) === true && ev(C.binary("||", 0n, 0.0)) === false);
  assert(ev(C.conditional(C.binary(">", x, 0n), C.binary("/", 10n, x), 0n), { x: 0n }) === 0n && ev(C.conditional(1n, 2n, 3.5)) === 2n); // the chosen one
  assert(ev(C.cast("uint8_t", 300n)) === 44n && ev(C.cast("int8_t", 200n)) === -56n && ev(C.cast("int", 2.9)) === 2n && ev(C.cast("int", -2.9)) === -2n);
  assert(ev(C.cast("bool", 0.5)) === true && ev(C.cast("bool", 0n)) === false && ev(C.cast("bool", -0)) === false);
  assert(ev(C.cast("double", 2n ** 60n + 1n)) === 2 ** 60 && ev(C.cast("float", 0.1)) === 0.10000000149011612 && ev(C.cast("int", true)) === 1n);
  assert(ev(C.cast("long double", 0.1)) === "0.1000000000000000055511151231257827" && ev(C.cast("double", C.cast("float", 0.1))) === 0.10000000149011612);
  assert(ev(C.constant(5n, "uint8_t")) === 5n && ev(C.constant("hi")) === "hi");
  // What C leaves undefined throws.
  const undefinedBehavior: [unknown, Function, string][] = [
    [C.binary("+", 2147483647n, 1n), FE.OverflowError, "signed overflow in +: 2147483648 does not fit 'int'"],
    [C.unary("-", -2147483648n), FE.OverflowError, "signed overflow in -: 2147483648 does not fit 'int'"],
    [C.binary("/", -2147483648n, -1n), FE.OverflowError, "signed overflow in /: 2147483648 does not fit 'int'"],
    [C.binary("%", 1n, 0n), FE.ZeroDivisionError, "integer division by zero"],
    [C.binary("<<", 1n, 32n), ValueError, "shift by 32 is undefined for 'int'"], [C.binary(">>", 1n, -1n), ValueError, "shift by -1 is undefined for 'int'"],
    [C.binary("<<", -1n, 1n), ValueError, "left shift of a negative value is undefined"],
    [C.binary("<<", 1n, 31n), FE.OverflowError, "signed overflow in <<: 2147483648 does not fit 'int'"],
    [C.cast("uint8_t", 300.0), FE.OverflowError, "300.0 is out of range of 'uint8_t'"],
    [C.cast("int", Infinity), FE.OverflowError, "inf is out of range of 'int'"],
    [C.cast("int", C.cast("long double", 1e300)), FE.OverflowError, " is out of range of 'int'"],
    [C.cast("int", NaN), ValueError, "NaN has no integer value"],
    [C.binary("%", 1.5, 1n), TypeError, "invalid operands to binary % ('double' and 'double')"],
    [C.binary("<<", 1.5, 1n), TypeError, "invalid operands to binary << ('double' and 'int')"],
    [C.unary("~", 1.5), TypeError, "invalid argument type 'double' to unary expression"],
    [C.binary("+", "a", 1n), TypeError, "invalid operand to +: 'str'"], [C.unary("!", "a"), TypeError, "invalid operand to !: 'str'"],
  ];
  for (const [expression, error, message] of undefinedBehavior) raises(error, () => ev(expression), message);
  // A variable's type is its value's (a value carries no domain: one of another type waits for domain inference).
  raises(FE.OverflowError, () => ev(x, { x: 2n ** 64n }), "integer 18446744073709551616 is too large for any type");
  assert(ev(C.cast("uint8_t", C.binary("+", x, 1n)), { x: 255n }) === 0n && ev(C.binary("+", x, x), { x: 1.5 }) === 3.0);
  assert(ev(C.binary("+", x, 1n), { x: 2n ** 40n }) === 2n ** 40n + 1n);
  assert(ev(C.binary("*", x, 2n), { x: -5n }) === -10n && ev(C.constant(5n, "word")) === 5n); // no type: its value's
  // Members, elements and calls.
  assert(ev(C.member(x, "age"), { x: ann }) === 30n && ev(C.member(x, "age", "->"), { x: { age: 5n } }) === 5n);
  assert(ev(C.member(x, "age"), { x: new Map([["age", 7n]]) }) === 7n);
  assert(ev(C.subscript(x, 1n), { x: [10n, 20n] }) === 20n && ev(C.subscript(C.subscript(C.member(x, "grid"), 1n), 0n), { x: board }) === 3n);
  const refused: [unknown, unknown, Function, string][] = [
    [C.member(x, "phone"), ann, KeyError, "no member named 'phone'"],
    [C.member(x, "a"), 1n, TypeError, "member reference base type 'int' is not a structure"],
    [C.subscript(x, 2n), [10n, 20n], ValueError, "index 2 is out of bounds of an array of 2"],
    [C.subscript(x, 0n), 1n, TypeError, "subscripted value is not an array: 'int'"],
    [C.subscript(x, 0.5), [1n], TypeError, "array subscript is not an integer: 'double'"],
    [C.call("f", 1n), null, FE.NameError, "use of undeclared identifier 'f'"], [C.identifier("z"), null, FE.NameError, "use of undeclared identifier 'z'"],
  ];
  for (const [expression, variable, error, message] of refused) raises(error, () => ev(expression, { x: variable }), message);
  assert(ev(C.member(x, "name"), { x: board }) === "b");
  const expression = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(C.binary("+", x, 1n))));
  assert(CText.ToText(expression) === "x + 1" && CV.OfAny(expression, { x: 41n }) === 42n);
  assert(CV.OfAny(cc(C.call("max", x, 3n)), new CV.Scope({ x: 5n }, { functions: { max: (a: bigint, b: bigint) => (a > b ? a : b) } })) === 5n);
}

<a id="DIA-13"></a>
## DIA-13 · SystemVerilog: expressions over 4-state vectors and reals, with IEEE 1800's precedence, sized literals and casts

In [ ]:
{
  const [x, y, v] = [SV.identifier("x"), SV.identifier("y"), SV.vector];
  const cases: [unknown, string][] = [
    [SV.binary("+", x, SV.binary("*", y, 2n)), "x + y * 2"], [SV.binary("*", SV.binary("+", x, y), 2n), "(x + y) * 2"],
    [SV.binary("-", x, SV.binary("-", y, 1n)), "x - (y - 1)"], [SV.binary("**", SV.binary("**", x, y), 2n), "x ** y ** 2"],
    [SV.binary("**", x, SV.binary("**", y, 2n)), "x ** (y ** 2)"], [SV.binary("<<", SV.binary("+", x, 1n), 2n), "x + 1 << 2"],
    [SV.binary("&&", SV.binary("||", x, y), x), "(x || y) && x"], [SV.binary("||", SV.binary("&&", x, y), x), "x && y || x"],
    [SV.binary("^~", SV.binary("|", x, y), SV.binary("&", x, y)), "(x | y) ^~ x & y"],
    [SV.binary("==", SV.binary("&", x, 1n), y), "(x & 1) == y"], [SV.binary("<", SV.binary(">>>", x, 1n), SV.binary("%", y, 2n)), "x >>> 1 < y % 2"],
    [SV.binary("->", SV.binary("->", x, y), x), "(x -> y) -> x"], [SV.binary("<->", x, SV.binary("->", y, x)), "x <-> y -> x"],
    [SV.binary("->", SV.conditional(x, y, 1n), y), "x ? y : 1 -> y"], [SV.conditional(SV.binary("->", x, y), 1n, 2n), "(x -> y) ? 1 : 2"],
    [SV.conditional(x, y, SV.conditional(y, 1n, 2n)), "x ? y : y ? 1 : 2"], [SV.conditional(SV.conditional(x, y, 1n), 2n, 3n), "(x ? y : 1) ? 2 : 3"],
    [SV.unary("-", SV.unary("-", x)), "-(-x)"], [SV.unary("&", SV.unary("&", x)), "&(&x)"], [SV.unary("~", SV.unary("!", x)), "~(!x)"],
    [SV.unary("-", -1n), "-(-1)"], [SV.binary("-", x, -1n), "x - -1"], [SV.unary("~&", SV.binary("+", x, 1n)), "~&(x + 1)"],
    [SV.inside(SV.binary("+", x, 1n), 1n, SV.span(2n, 5n)), "x + 1 inside {1, [2:5]}"], [SV.binary("<", SV.inside(x, 1n), 0n), "x inside {1} < 0"],
    [SV.binary("+", SV.inside(x, 1n), 1n), "(x inside {1}) + 1"], [SV.binary("==?", x, v("1x")), "x ==? 2'b1x"],
    [v("11111111", null, "h"), "8'hff"], [v("10x1", null, "h"), "4'b10x1"], [v("1xxxx", null, "h"), "5'h1x"], [v("1zzz", null, "h"), "4'b1zzz"],
    [v("00000101", true, "d"), "8'sd5"], [v("xxxx", null, "d"), "4'dx"], [v("0x", null, "d"), "2'b0x"], [v("111", null, "o"), "3'o7"],
    [v("10z1"), "4'b10z1"], [v("1", true), "1'sb1"],
    [SV.constant(1.5), "1.5"], [SV.constant(1.0), "1.0"], [SV.constant(1e300), "1e+300"], [SV.constant(NaN), "0.0 / 0.0"],
    [SV.constant(Infinity), "1.0 / 0.0"], [SV.constant(-Infinity), "-1.0 / 0.0"], [SV.binary("*", x, Infinity), "x * (1.0 / 0.0)"],
    [SV.unary("-", Infinity), "-(1.0 / 0.0)"], [SV.constant('a"\\\n\t'), '"a\\"\\\\\\n\\t"'], [SV.constant(-5n), "-5"],
    [SV.cast("signed", x), "signed'(x)"], [SV.cast(8n, SV.binary("+", x, 1n)), "8'(x + 1)"], [SV.cast("byte", -1n), "byte'(-1)"],
    [SV.member(SV.member(x, "a"), "b"), "x.a.b"], [SV.select(SV.binary("+", x, y), 0n), "(x + y)[0]"], [SV.range_(x, 7n, 4n), "x[7:4]"],
    [SV.concatenation(x, SV.replication(3n, y)), "{x, {3{y}}}"], [SV.call("$clog2", x), "$clog2(x)"], [SV.call("f", x, 1n), "f(x, 1)"],
  ];
  for (const [expression, source] of cases) assert(SVText.ToText(sv(expression)) === source, `${source}: ${SVText.ToText(sv(expression))}`);
  const sample = sv(SV.binary("->", SV.inside(SV.member(x, "a"), 1n, SV.span(2n, 5n)), SV.call("$clog2", SV.iterate(
    SV.method(x, "unique"), "sum", "q", SV.identifier("q")))));
  assert(roundTrips(sample) && SVText.ToText(sample) === "x.a inside {1, [2:5]} -> $clog2(x.unique().sum(q) with (q))");
  assert(SV.DIALECT.validate(sample, SY.free(sample, SV.DIALECT)).length === 0 && equal(F.form(sample).attributes, map({ operator: "->" })));
  // Validation: a vector's bits and base, and a cast's type or width.
  assert(same(SV.DIALECT.validate(sv(v("12"))), ["a vector's bits must be 0, 1, x and z, at least one, got '12'"]));
  assert(same(SV.DIALECT.validate(sv(v(""))), ["a vector's bits must be 0, 1, x and z, at least one, got ''"]));
  assert(same(SV.DIALECT.validate(sv(v("1", null, "q"))), ["a vector's base must be b, o, d or h, got 'q'"]));
  assert(same(SV.DIALECT.validate(sv(SV.cast("word", x)), ["x"]), ["a cast's type must be a built-in type, signed or unsigned, got 'word'"]));
  assert(same(SV.DIALECT.validate(sv(SV.cast(0n, x)), ["x"]), ["a cast's width must be positive, got 0"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.cast((r: any) => r.operand(x))), ["x"]), ["a cast names a type or a width, not both nor neither"]));
  assert(same(SV.DIALECT.validate(sv(SV.binary("<<=", x, 1n)), ["x"], true), ["'<<=' is not a core operation"]));
  assert(SV.DIALECT.validate(sv(SV.call("f", x)), ["x"], true).length === 0 && same(SV.DIALECT.validate(sv(SV.identifier("x"))), ["variable 'x' is not bound"]));
  assert(same(SV.DIALECT.validate(sv(SV.constant(true))), ["a SystemVerilog literal cannot hold a bool"])
    && same(SV.DIALECT.validate(sv(SV.constant(new Uint8Array([0])))), ["a SystemVerilog literal cannot hold a bytes"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.literal((r: any) => r))), ["a literal needs a value"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.method((r: any) => r.array(x))), ["x"]), [
    "a method needs a name", "an array method must be one of size, sum, product, and, or, xor, min, max, unique, got None"]));
  assert(same(SV.DIALECT.validate(sv((u: any) => u.iterate((r: any) => r.name("p").array(x).body(SV.identifier("p")))), ["x"]), [
    "an iterate needs a method", "an iteration's method must be one of sum, product, and, or, xor, got None",
    "body: variable 'p' is not bound"]));
  raises(TypeError, () => SV.DIALECT.spec(true), "expected a term, a writer or a native value, got True"); // a bool is no SystemVerilog constant
  raises(ValueError, () => SVText.ToText(term(Patterns.Expressions.let_("a", 1n, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT)), "a Patterns literal is not a SystemVerilog term"); // its terms first
  // Types: integral vectors of a width, a signedness and 2 or 4 states, and reals.
  const T = (name: string) => SVD.TYPES.get(name) as SystemVerilog.Domains.SvType;
  const widths = ["bit", "logic", "reg", "byte", "shortint", "int", "longint", "integer", "time"].map((n) => T(n).width);
  assert(same(widths, [1, 1, 1, 8, 16, 32, 64, 32, 64]));
  assert(T("int").signed && T("int").states === 2 && T("integer").states === 4 && !T("time").signed && T("logic").equals(T("reg")));
  assert(T("int").name() === "int" && String(T("integer")) === "integer" && SVD.vector(8).name() === "logic [7:0]");
  assert(SVD.vector(4, true, 2).name() === "bit signed [3:0]" && SVD.vector(32, true, 2).equals(T("int")) && T("shortreal").format === "binary32");
  assert(SVD.common([T("byte"), T("int")]).equals(T("int")) && SVD.common([T("byte"), SVD.vector(16)]).equals(SVD.vector(16)));
  assert(SVD.common([T("shortreal"), T("int")]).equals(T("real")) && SVD.common([T("shortreal"), T("shortreal")]).equals(T("shortreal")));
  assert(SVD.common([T("bit"), T("logic")]).equals(T("logic")));
  const ofs = [1n, -(2n ** 31n), 2n ** 31n, 2n ** 63n, 1.5, "a", true].map((value) => SVD.of(value));
  const expectedOfs = [T("integer"), T("integer"), T("longint"), D.Anything, T("real"), D.Anything, D.Anything];
  assert(ofs.every((domain, i) => domain === expectedOfs[i] || (expectedOfs[i] as any).equals(domain)));
  // Values: a Logic is a type and its bits, encoded as VPI does.
  const a = SVD.Logic.of("10xz");
  assert(a.aval === 0b1010n && a.bval === 0b0011n && a.bits === "10xz" && !a.known && String(a) === "4'b10xz");
  assert(SVD.Logic.of("10xz", true, 2).bits === "1000" && String(SVD.Logic.of("1", true)) === "1'sb1");
  assert(SVD.Logic.of("1110", true).integer() === -2n && SVD.Logic.of("1110").integer() === 14n && SVD.Logic.number(T("byte"), -1n).bits === "11111111");
  raises(ValueError, () => a.integer(), "4'b10xz has unknown bits");
  assert(T("int").contains(SVD.Logic.number(T("int"), 5n)) && !T("int").contains(SVD.Logic.number(T("integer"), 5n)) && !T("int").contains(5n));
  assert(T("real").contains(1.5) && !T("real").contains(1n) && T("int").includes(SVD.vector(32, true, 2)) && !T("int").includes(T("integer")));
  assert(T("int").includes(new D.OfUnion(T("int"))) && !T("int").includes(D.Anything));
  // Inference sizes an expression as IEEE 1800 does.
  const infer = (expression: unknown, environment?: Record<string, any>) => SV.DIALECT.infer(sv(expression), environment) as any;
  const is = (domain: unknown, expected: SystemVerilog.Domains.SvType) => expected.equals(domain);
  assert(is(infer(SV.binary("+", v("1010"), 1n)), SVD.vector(32)) && is(infer(SV.binary("+", 1n, 2n)), T("integer")));
  assert(is(infer(SV.binary("+", SV.cast("byte", 1n), SV.cast("int", 1n))), T("int")) && is(infer(SV.binary("*", 1n, 1.5)), T("real")));
  assert(is(infer(SV.binary("**", v("1010"), 2n)), SVD.vector(4)) && is(infer(SV.binary("**", 2n, 0.5)), T("real")));
  assert(is(infer(SV.binary("<<", v("1010"), 40n)), SVD.vector(4)) && is(infer(SV.binary("==", v("1010"), 1n)), SVD.vector(1)));
  assert(is(infer(SV.binary("===", v("1010"), 1n)), T("bit")) && is(infer(SV.binary("==?", v("1010"), 1n)), SVD.vector(1)));
  assert(is(infer(SV.binary("&&", x, 1n), { x: D.Anything }), SVD.vector(1)) && infer(SV.binary("+", x, 1n), { x: D.Anything }) === D.Anything);
  assert(is(infer(SV.binary("&", v("10"), v("1000"))), SVD.vector(4)) && is(infer(SV.unary("~", SV.cast("byte", 1n))), T("byte")));
  assert(is(infer(SV.unary("-", 1.5)), T("real")) && is(infer(SV.unary("!", 1.5)), SVD.vector(1)) && is(infer(SV.unary("^", v("10"))), SVD.vector(1)));
  assert(is(infer(SV.binary("%", 7n, 2n)), T("integer")) && is(infer(SV.cast("int", x), { x: D.Anything }), T("int")));
  assert(infer(SV.cast(8n, x), { x: D.Anything }) === D.Anything && infer(SV.cast("signed", 1n)) === D.Anything); // sized by the operand
  assert(is(infer(SV.inside(x, 1n), { x: D.Anything }), SVD.vector(1)) && infer(SV.conditional(x, 1n, 2n), { x: D.Anything }) === D.Anything);
  assert(SV.DIALECT.kinds().get("vector")!.typed!(F.kindOf(sv(v("12")))[1]) === null && is(infer(v("10x1", true)), SVD.vector(4, true)) && is(infer(SV.constant(1.5)), T("real")));
  assert(infer(SV.binary("+", "a", 1n)) === D.Anything); // a string has no type of its own
  raises(TypeError, () => infer(SV.binary("+", x, 1n), { x: new D.OfUnion(T("int"), T("real")) }),
    "+ cannot take (int | real, integer); it takes (T, T) -> their common type");
  assert(SVD.BINARY.get("+")?.describe() === "(T, T) -> their common type" && SVD.UNARY.get("!")?.arity() === 1);
  const refused: [unknown, string][] = [
    [SV.binary("%", 1.5, 1n), "% cannot take (real, integer); it takes (integral, integral) -> their common type"],
    [SV.binary("&", 1.5, 1n), "& cannot take (real, integer)"], [SV.binary("<<", 1.5, 1n), "<< cannot take (real, integer)"],
    [SV.binary("===", 1.5, 1n), "=== cannot take (real, integer)"], [SV.binary("==?", 1.5, 1n), "==? cannot take (real, integer)"],
    [SV.unary("~", 1.5), "~ cannot take (real); it takes (integral) -> its type"], [SV.unary("&", 1.5), "& cannot take (real)"],
  ];
  for (const [expression, message] of refused) raises(TypeError, () => infer(expression), message);
}

<a id="DIA-14"></a>
## DIA-14 · SystemVerilog evaluates by IEEE 1800's rules: sizing by context, 4-state logic, casts and system functions

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => SVV.OfAny(sv(expression), variables) as any;
  const [v, b, u, c, call] = [SV.vector, SV.binary, SV.unary, SV.cast, SV.call];
  const x = SV.identifier("x");
  const T = (name: string) => SVD.TYPES.get(name) as SystemVerilog.Domains.SvType;
  // Each expression and its value: a Logic by its text, or a real.
  const cases: [unknown, string | number][] = [
    [b("+", v("1010"), v("0110")), "4'b0000"], [b("+", v("1111"), 1n), "32'b" + "0".repeat(27) + "10000"],
    [b("/", -7n, 2n), "32'sb" + "1".repeat(30) + "01"], [b("%", -7n, 2n), "32'sb" + "1".repeat(32)],
    [b("/", 1n, 0n), "32'sb" + "x".repeat(32)], [b(">>>", v("11111110", true), 1n), "8'sb11111111"], [b(">>>", v("1000"), 1n), "4'b0100"],
    [b(">>", v("1000", true), 1n), "4'sb0100"], [b("&", v("10x1"), v("0011")), "4'b00x1"], [b("|", v("10z1"), v("1100")), "4'b11x1"],
    [b("^", v("10x1"), v("0011")), "4'b10x0"], [b("~^", v("1001"), v("0011")), "4'b0101"], [b("+", v("10x1"), 1n), "32'b" + "x".repeat(32)],
    [b("==", v("10x1"), v("10x1")), "1'bx"], [b("===", v("10x1"), v("10x1")), "1'b1"], [b("!==", v("10z1"), v("10x1")), "1'b1"],
    [b("==?", v("1001"), v("1xx1")), "1'b1"], [b("!=?", v("1001"), v("1xx0")), "1'b1"], [b("==?", v("1x01"), v("1001")), "1'bx"],
    [u("&", v("1111")), "1'b1"], [u("|", v("000x")), "1'bx"], [u("^", v("1011")), "1'b1"], [u("~&", v("1101")), "1'b1"],
    [u("~|", v("0000")), "1'b1"], [u("&", v("1x11")), "1'bx"], [u("|", v("0100")), "1'b1"], [u("~^", v("1x11")), "1'bx"], [u("~^", v("1011")), "1'b0"], [u("^", v("1x11")), "1'bx"], [u("&", v("0x11")), "1'b0"],
    [u("-", v("0001")), "4'b1111"], [u("~", v("10x1")), "4'b01x0"], [u("-", v("0x01")), "4'bxxxx"], [u("+", v("0x01")), "4'b0x01"],
    [b("**", 3n, -1n), "32'sb" + "0".repeat(32)], [b("**", 0n, -1n), "32'sb" + "x".repeat(32)],
    [b("**", -1n, -1n), "32'sb" + "1".repeat(32)], [b("**", -1n, -2n), "32'sb" + "0".repeat(31) + "1"],
    [b("**", 2n, 10n), "32'sb" + "0".repeat(21) + "10000000000"], [b("**", 1n, -3n), "32'sb" + "0".repeat(31) + "1"],
    [b("**", 2.0, 0.5), 2 ** 0.5], [b("/", 1.0, 0n), Infinity], [b("**", 2n, 0.5), 2 ** 0.5], [b("+", 1n, 0.5), 1.5],
    [SV.conditional(v("x"), v("1100"), v("1010")), "4'b1xx0"], [SV.conditional(v("1"), v("1100"), v("1010")), "4'b1100"],
    [SV.conditional(0n, v("1100"), v("1010")), "4'b1010"], [SV.conditional(v("z"), 1.5, 1.5), 1.5], [SV.conditional(v("z"), 1.5, 2.5), 0.0],
    [SV.inside(3n, 1n, SV.span(2n, 5n)), "1'b1"], [SV.inside(7n, 1n, SV.span(2n, 5n)), "1'b0"], [SV.inside(v("1x00"), v("1100")), "1'bx"],
    [SV.inside(v("1x00"), v("1100"), v("1x00")), "1'b1"], [SV.inside(v("1x00"), SV.span(0n, 3n)), "1'bx"],
    [SV.inside(v("1x00"), SV.span(8n, 15n)), "1'bx"], [SV.inside(v("1100"), v("1x00")), "1'b1"], [c("byte", 300n), "8'sb00101100"],
    [c("byte", 200n), "8'sb11001000"], [c("unsigned", -1n), "32'b" + "1".repeat(32)], [c(4n, v("11111111")), "4'b1111"],
    [c("int", 2.5), "32'sb" + "0".repeat(30) + "11"], [c("int", -2.5), "32'sb" + "1".repeat(30) + "01"], [c("real", 3n), 3.0],
    [c("shortreal", 0.1), 0.10000000149011612], [c("int", v("10x1")), "32'sb" + "0".repeat(28) + "1001"], [c("bit", v("x")), "1'b0"],
    [c("signed", v("1111")), "4'sb1111"], [c(8n, v("1111", true)), "8'sb11111111"], [c(8n, v("z1", true)), "8'sbzzzzzzz1"],
    [c(4n, v("x0", true)), "4'sbxxx0"], [call("$clog2", 8n), "32'sb" + "0".repeat(30) + "11"],
    [call("$clog2", 9n), "32'sb" + "0".repeat(29) + "100"], [call("$clog2", 0n), "32'sb" + "0".repeat(32)],
    [call("$clog2", v("1x")), "32'sb" + "x".repeat(32)], [call("$bits", v("00000000")), "32'sb" + "0".repeat(28) + "1000"],
    [call("$bits", 1.5), "32'sb" + "0".repeat(25) + "1000000"], [call("$countones", v("1011")), "32'sb" + "0".repeat(30) + "11"],
    [call("$onehot", v("0100")), "1'b1"], [call("$onehot", v("0000")), "1'b0"], [call("$onehot0", v("0000")), "1'b1"],
    [call("$isunknown", v("10x1")), "1'b1"], [call("$isunknown", v("1001")), "1'b0"], [call("$signed", v("1111")), "4'sb1111"],
    [call("$unsigned", v("1111", true)), "4'b1111"], [SV.concatenation(v("1010"), v("11")), "6'b101011"],
    [SV.replication(2n, v("10")), "4'b1010"], [SV.concatenation(v("x"), v("1")), "2'bx1"], [SV.select(v("1010"), 1n), "1'b1"],
    [SV.select(v("1010"), 4n), "1'bx"], [SV.select(v("1010"), v("x")), "1'bx"], [SV.range_(v("1010"), 3n, 2n), "2'b10"],
    [SV.range_(v("1010"), 5n, 2n), "4'bxx10"], [SV.select(c("int", v("1010")), 40n), "1'b0"],
    [b("+", v("1111", true), v("00000000", true)), "8'sb11111111"], [b("+", v("1111", true), v("00000000")), "8'b00001111"],
    [b("<", -1n, v("1")), "1'b0"], [b("<", -1n, 1n), "1'b1"], [b("==", b("+", v("1111"), v("0001")), v("10000")), "1'b1"],
    [b("==", b("<<", v("1000"), 1n), v("10000")), "1'b1"], [b("<<", v("0001"), v("10")), "4'b0100"],
    [b("<<", v("0001"), v("x")), "4'bxxxx"], [b("+", b("<", 2n, 1n), v("0001")), "4'b0001"], [b("+", b("<", 1n, 2n), v("0001")), "4'b0010"],
    [u("!", v("0000")), "1'b1"], [u("!", v("00x0")), "1'bx"], [u("!", 1.5), "1'b0"], [u("!", 0.0), "1'b1"], [b(">>>", v("x000", true), 1n), "4'sbxx00"], [b("&&", v("x"), 0n), "1'b0"],
    [b("||", v("x"), 1n), "1'b1"], [b("&&", v("x"), 1n), "1'bx"], [b("&&", 1n, 1n), "1'b1"], [b("||", 0n, 0n), "1'b0"], [b("&&", 0n, v("x")), "1'b0"],
    [b("||", 1n, v("x")), "1'b1"], [b("->", 0n, v("x")), "1'b1"], [b("->", 1n, v("x")), "1'bx"], [b("->", 1n, 0n), "1'b0"],
    [b("->", v("x"), 1n), "1'b1"], [b("<->", 1n, 1n), "1'b1"], [b("<->", 1n, 0n), "1'b0"], [b("<->", v("x"), 0n), "1'bx"],
    [b(">", 1.5, 1n), "1'b1"], [b("==", NaN, NaN), "1'b0"], [b("!=", NaN, 1.0), "1'b1"], [b("<", v("x"), 1n), "1'bx"], [u("-", 1.5), -1.5],
    [u("+", 1.5), 1.5], [b("-", c("shortreal", 1.5), c("shortreal", 0.25)), 1.25], [b("*", 3n, 4n), "32'sb" + "0".repeat(28) + "1100"],
    [b("-", 3n, 4n), "32'sb" + "1".repeat(32)], [b("%", 7n, -2n), "32'sb" + "0".repeat(31) + "1"],
  ];
  for (const [expression, value] of cases) {
    const result = ev(expression);
    assert(typeof value === "string" ? String(result) === value : result === value, `${SVText.ToText(sv(expression))}: ${result}`);
  }
  assert(Number.isNaN(ev(b("**", -8.0, 1 / 3))) && ev(b("**", 0.0, -1.0)) === Infinity && ev(b("**", 10.0, 400.0)) === Infinity);
  assert(ev(b("**", -10.0, 401.0)) === -Infinity && ev(b("**", -10.0, 400.0)) === Infinity && ev(b("**", c("shortreal", 2n), c("shortreal", 0.5))) === 1.4142135381698608);
  assert(ev(b("**", -0, -1.0)) === -Infinity && ev(b("**", -0, -2.0)) === Infinity && ev(b("**", 0.0, -3.0)) === Infinity);
  assert(ev(b("**", 1.0, NaN)) === 1.0 && ev(b("**", -1.0, Infinity)) === 1.0 && ev(b("**", NaN, 0.0)) === 1.0); // as C's pow
  assert(String(ev(b("/", c("int", 1n), c("int", 0n)))) === "32'sb" + "0".repeat(32) && String(ev(b("&", c("int", 5n), v("x")))) === "32'b" + "0".repeat(31) + "x");
  // Variables are typed by their values, Patterns' typed values among them.
  assert(String(ev(x, { x: true })) === "1'b1" && ev(x, { x: 5n }).type.equals(T("integer")) && ev(x, { x: 2n ** 40n }).type.equals(T("longint")));
  assert(String(ev(b("<<", 1n, v("1".repeat(64))))) === "32'sb" + "0".repeat(32) && String(ev(b(">>>", -4n, v("1".repeat(64))))) === "32'sb" + "1".repeat(32));
  assert(String(ev(call("$bits", c("shortreal", 1.5)))) === "32'sb" + "0".repeat(26) + "100000" && String(ev(x, { x: "é" })) === "16'b1100001110101001");
  assert(String(ev(x, { x: "AB" })) === "16'b0100000101000010" && String(ev(x, { x: "" })) === "8'b00000000" && ev(x, { x: 1.5 }) === 1.5);
  const record = new Map([["a", 1n]]);
  assert(String(ev(x, { x: SVD.Logic.of("1x") })) === "2'b1x" && ev(x, { x: record }) === record); // a value carries no domain
  assert(String(ev(SV.member(x, "age"), { x: ann })) === "32'sb" + "0".repeat(26) + "011110" && String(ev(SV.member(x, "a"), { x: { a: true } })) === "1'b1");
  assert(String(ev(SV.select(x, 1n), { x: [1n, 2n] })) === "32'sb" + "0".repeat(30) + "10"
    && String(ev(SV.select(SV.member(x, "a"), 0n), { x: new Map([["a", [false]]]) })) === "1'b0");
  assert(String(ev(SV.select(SV.member(x, "ports"), 1n), { x: bus })) === "32'sb" + "0".repeat(29) + "101"); // data objects in the store
  const f = (a: any, b: any) => a.integer() + b.integer();
  assert((SVV.OfAny(sv(SV.call("f", x, 1n)), new SVV.Scope({ x: 2n }, { functions: { f } })) as any).type.equals(T("integer")));
  assert((SVV.OfAny(sv(b("+", 1n, 1n)), new SVV.Scope()) as any).integer() === 2n && (SVV.OfAny(sv(SV.binary("+", 1n, 2n))) as any).integer() === 3n);
  const expression = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(SV.binary("+", x, 1n))));
  assert((SVV.OfAny(expression, { x: 2n }) as any).integer() === 3n && SVText.ToText(expression) === "x + 1"); // evaluated from its root
  const errors: [[unknown, Record<string, unknown>], Function, string][] = [
    [[x, {}], NameError, "identifier 'x' is not declared"], [[SV.call("f", 1n), {}], NameError, "function 'f' is not declared"],
    [[v("12"), {}], ValueError, "cannot evaluate an invalid expression: a vector's bits must be 0, 1, x and z, at least one, got '12'"],
    [[x, { x: 2n ** 70n }], OverflowError, "integer 1180591620717411303424 is too large for any type"],
    [[u("~", 1.5), {}], TypeError, "~ expects an integral operand, got real"],
    [[u("&", 1.5), {}], TypeError, "the reduction & expects an integral operand, got real"],
    [[b("%", 1.5, 1n), {}], TypeError, "% expects integral operands, got real"],
    [[b("&", 1.5, 1n), {}], TypeError, "& expects integral operands, got real"],
    [[b("<<", 1.5, 1n), {}], TypeError, "<< expects an integral operand, got real"],
    [[b("<<", 1n, 1.5), {}], TypeError, "<< expects integral operands, got real"],
    [[b("+", x, 1n), { x: new Map() }], TypeError, "+ expects a number, got dict"],
    [[c("int", x), { x: new Map() }], TypeError, "a cast expects a number, got dict"],
    [[SV.conditional(1n, x, 1n), { x: new Map() }], TypeError, "?: expects a number, got dict"],
    [[u("!", x), { x: new Map() }], TypeError, "a dict has no truth value"],
    [[c("signed", 1.5), {}], TypeError, "signed' expects an integral operand, got real"],
    [[c("int", NaN), {}], OverflowError, "nan has no integer value"],
    [[c("int", -Infinity), {}], OverflowError, "-inf has no integer value"],
    [[SV.member(x, "b"), { x: new Map([["a", 1n]]) }], KeyError, "no member named 'b'"],
    [[SV.member(x, "b"), { x: 1n }], TypeError, "a Logic has no members"],
    [[SV.select(x, 2n), { x: [1n, 2n] }], ValueError, "index 2 is out of bounds of an array of 2"],
    [[SV.select(x, v("x")), { x: [1n, 2n] }], ValueError, "an array's index must be known, got 1'bx"],
    [[SV.select(x, 0n), { x: new Map([["a", 1n]]) }], TypeError, "a dict cannot be selected from"],
    [[SV.range_(x, 1n, 0n), { x: new Map() }], TypeError, "a dict cannot be selected from"],
    [[SV.range_(v("1010"), 0n, 1n), {}], ValueError, "a range's msb must be at least its lsb, got [0:1]"],
    [[SV.range_(v("1010"), v("x"), 0n), {}], ValueError, "a range's msb must be a known integer, got 1'bx"],
    [[SV.range_(v("1010"), 1n, 0.5), {}], ValueError, "a range's lsb must be a known integer, got 0.5"],
    [[SV.replication(0n, v("1")), {}], ValueError, "a replication's count must be positive, got 0"],
    [[SV.concatenation(v("1"), 1.5), {}], TypeError, "a concatenation's parts must be integral, got float"],
    [[call("$clog2", 1n, 2n), {}], TypeError, "$clog2 takes 1 argument, got 2"],
    [[call("$signed", 1.5), {}], TypeError, "$signed expects an integral argument, got real"],
    [[call("$bits", x), { x: new Map() }], TypeError, "$bits expects a number, got dict"],
    [[call("$countones", 1.5), {}], TypeError, "$countones expects an integral argument, got real"],
  ];
  for (const [[expression, variables], error, message] of errors) raises(error, () => ev(expression, variables), message);
}

<a id="DIA-15"></a>
## DIA-15 · SystemVerilog arrays: methods, reductions with a `with` clause, and `inside` over an array's elements

In [ ]:
{
  const [xs, p, q] = [SV.identifier("xs"), SV.identifier("p"), SV.identifier("q")];
  assert(SVText.ToText(sv(SV.iterate(SV.member(xs, "pins"), "and", "p", SV.binary(">", p, 0n)))) === "xs.pins.and(p) with (p > 0)");
  assert(SVText.ToText(sv(SV.select(SV.method(xs, "min"), 0n))) === "xs.min()[0]" && SVText.ToText(sv(SV.method(SV.binary("+", xs, 1n), "size"))) === "(xs + 1).size()");
  assert(SVText.ToText(sv(SV.inside(1n, xs, SV.span(2n, 3n)))) === "1 inside {xs, [2:3]}");
  assert(same(SV.DIALECT.validate(sv(SV.method(xs, "pop")), ["xs"]), [
    "an array method must be one of size, sum, product, and, or, xor, min, max, unique, got 'pop'"]));
  assert(same(SV.DIALECT.validate(sv(SV.iterate(xs, "min", "p", p)), ["xs"]), [ // a malformed iteration binds no name
    "an iteration's method must be one of sum, product, and, or, xor, got 'min'", "body: variable 'p' is not bound"]));
  assert(SV.DIALECT.validate(sv(SV.iterate(xs, "and", "p", p)), ["xs"]).length === 0
    && same([...SY.free(sv(SV.iterate(xs, "and", "p", SV.binary("&", p, q))), SV.DIALECT)].sort(), ["q", "xs"]));
  assert(SV.DIALECT.infer(sv(SV.iterate(xs, "and", "p", p)), { xs: D.Anything }) === D.Anything);
  assert(SV.DIALECT.infer(sv(SV.method(xs, "size")), { xs: D.Anything }) === D.Anything);
  const ev = (expression: unknown, variables: Record<string, unknown> = {}) => SVV.OfAny(sv(expression), variables) as any;
  const data = [1n, 2n, 3n, 2n];
  const cases: [unknown, string][] = [ // each over [1, 2, 3, 2], and its value
    [SV.method(xs, "size"), "32'sb" + "0".repeat(29) + "100"], [SV.method(xs, "sum"), "32'sb" + "0".repeat(28) + "1000"],
    [SV.method(xs, "product"), "32'sb" + "0".repeat(28) + "1100"], [SV.method(xs, "and"), "32'sb" + "0".repeat(32)],
    [SV.method(xs, "or"), "32'sb" + "0".repeat(30) + "11"], [SV.method(xs, "xor"), "32'sb" + "0".repeat(30) + "10"],
    [SV.select(SV.method(xs, "min"), 0n), "32'sb" + "0".repeat(31) + "1"], [SV.select(SV.method(xs, "max"), 0n), "32'sb" + "0".repeat(30) + "11"],
    [SV.method(SV.method(xs, "unique"), "size"), "32'sb" + "0".repeat(30) + "11"],
    [SV.iterate(xs, "and", "p", SV.binary(">", p, 0n)), "1'b1"], [SV.iterate(xs, "and", "p", SV.binary(">", p, 1n)), "1'b0"],
    [SV.iterate(xs, "or", "p", SV.binary(">", p, 2n)), "1'b1"], [SV.iterate(xs, "xor", "p", SV.binary("==", p, 2n)), "1'b0"],
    [SV.iterate(xs, "sum", "p", SV.cast("int", SV.binary(">=", p, 2n))), "32'sb" + "0".repeat(30) + "11"],
    [SV.iterate(xs, "sum", "p", SV.binary("*", p, p)), "32'sb" + "0".repeat(27) + "10010"],
    [SV.iterate(xs, "product", "p", SV.binary("+", p, 1n)), "32'sb" + "0".repeat(25) + "1001000"],
    [SV.inside(3n, xs), "1'b1"], [SV.inside(5n, xs, 7n), "1'b0"], [SV.inside(7n, xs, 7n), "1'b1"], [SV.inside(2n, SV.span(0n, 1n), xs), "1'b1"],
  ];
  for (const [expression, value] of cases) assert(String(ev(expression, { xs: data })) === value, SVText.ToText(sv(expression)));
  // x and z reduce as the operators take them: and() is 0 if any value is, x if any is unknown and none 0.
  assert(String(ev(SV.iterate(xs, "and", "p", p), { xs: [SVD.Logic.of("1"), SVD.Logic.of("x")] })) === "1'bx");
  assert(String(ev(SV.iterate(xs, "and", "p", p), { xs: [SVD.Logic.of("0"), SVD.Logic.of("x")] })) === "1'b0");
  assert(String(ev(SV.inside(SV.vector("1x"), xs), { xs: [SVD.Logic.of("10")] })) === "1'bx");
  assert(String(ev(SV.method(SV.method(xs, "unique"), "size"), { xs: [SVD.Logic.of("1x"), SVD.Logic.of("1x"), 1.0, 1n, 1.0] })) === "32'sb" + "0".repeat(30) + "11");
  // Of no items, the reductions give their identities, and the locators no item.
  assert(same(["sum", "product", "and", "or", "xor"].map((m) => String(ev(SV.method(xs, m), { xs: [] }))), [
    "32'sb" + "0".repeat(32), "32'sb" + "0".repeat(31) + "1", "1'b1", "1'b0", "1'b0"]));
  assert(ev(SV.method(xs, "min"), { xs: [] }).length === 0 && String(ev(SV.iterate(xs, "and", "p", SV.binary(">", p, 0n)), { xs: [] })) === "1'b1");
  assert(ev(SV.method(xs, "sum"), { xs: [1.5, 2n] }) === 3.5 && String(ev(SV.select(SV.method(xs, "max"), 0n), { xs: [1.5, 2n] })) === "32'sb" + "0".repeat(30) + "10");
  const errors: [unknown, Record<string, unknown>, Function, string][] = [
    [SV.method(xs, "size"), { xs: 5n }, TypeError, "size() is a method of arrays, not of Logic"],
    [SV.iterate(xs, "and", "p", p), { xs: new Map([["a", 1n]]) }, TypeError, "and() is a method of arrays, not of dict"],
    [SV.method(xs, "sum"), { xs: [new Map([["a", 1n]])] }, TypeError, "an array's elements must be numbers, got dict"],
    [SV.inside(1n, xs), { xs: [new Map([["a", 1n]])] }, TypeError, "an array's elements must be numbers, got dict"],
    [SV.inside(1n, xs), { xs: new Map([["a", 1n]]) }, TypeError, "inside expects a number, got dict"],
    [SV.method(xs, "and"), { xs: [1.5, 2.5] }, TypeError, "& expects integral operands, got real"],
  ];
  for (const [expression, variables, error, message] of errors) raises(error, () => ev(expression, variables), message);
}

<a id="DIA-16"></a>
## DIA-16 · Matlab arrays: arrayfun, indexing from 1, and the functions of arrays

In [ ]:
{
  const [xs, p] = [M.identifier("xs"), M.identifier("p")];
  assert(MText.ToText(mat(M.call("all", M.arrayfun("p", M.field(xs, "pins"), M.binary(">", p, 0n))))) === "all(arrayfun(@(p) p > 0, xs.pins))");
  assert(MText.ToText(mat(M.index(xs, M.binary("+", p, 1n)))) === "xs(p + 1)" && MText.ToText(mat(M.index(M.field(xs, "a"), 2n))) === "xs.a(2)");
  assert(same(M.DIALECT.validate(mat(M.arrayfun("_p", xs, 1n)), ["xs"]), ["an arrayfun's parameter must be an identifier, got '_p'"]));
  assert(M.DIALECT.validate(mat(M.arrayfun("p", xs, p)), ["xs"]).length === 0
    && same([...SY.free(mat(M.arrayfun("p", xs, M.binary("+", p, M.identifier("q")))), M.DIALECT)].sort(), ["q", "xs"]));
  assert(same(M.DIALECT.validate(mat(M.call("numel", xs, 1n)), ["xs"]), ["numel takes 1 arguments, got 2"]));
  assert(M.DIALECT.infer(mat(M.call("all", xs)), { xs: D.Anything }).name() === "logical");
  assert(M.DIALECT.infer(mat(M.arrayfun("p", xs, p)), { xs: D.Anything }) === D.Anything);
  assert(MText.ToText(mat(M.array(1n, xs, M.array()))) === "[1, xs, []]" && M.DIALECT.validate(mat(M.array(1n, xs)), ["xs"]).length === 0);
  assert(equal(MV.OfAny(mat(M.array(1n, xs)), { xs: [true, 2n] }), [1, 1, 2]) && equal(MV.OfAny(mat(M.array(true, false))), [true, false]));
  assert(equal(MV.OfAny(mat(M.array("a", xs)), { xs: ["b"] }), ["a", "b"])); // a string array
  raises(TypeError, () => MV.OfAny(mat(M.array(1n, "a"))), "Concatenation takes numbers and logicals, or strings.");
  const alike = (a: any, b: any): boolean => (Array.isArray(b)
    ? Array.isArray(a) && a.length === b.length && a.every((x, i) => alike(x, b[i]))
    : typeof a === typeof b && (a === b || (Number.isNaN(a) && Number.isNaN(b))));
  const VALUES: [any, unknown, unknown][] = [
    [M.call("all", M.arrayfun("p", xs, M.binary(">", p, 0n))), [1n, 2n, 3n, 2n], true],
    [M.call("any", M.arrayfun("p", xs, M.binary(">", p, 2n))), [1n, 2n, 3n, 2n], true],
    [M.call("nnz", M.arrayfun("p", xs, M.binary(">=", p, 2n))), [1n, 2n, 3n, 2n], 3.0], [M.call("numel", xs), [1n, 2n, 3n, 2n], 4.0],
    [M.index(xs, 2n), [1n, 2n, 3n, 2n], 2.0], [M.index(xs, true), [1n, 2n, 3n, 2n], 1.0], [M.index(xs, false), [1n, 2n, 3n, 2n], []],
    [M.call("ismember", 3n, xs), [1n, 2n, 3n, 2n], true], [M.call("ismember", "3", xs), [1n, 2n, 3n, 2n], true],
    [M.call("ismember", 5n, xs), [1n, 2n, 3n, 2n], false], [M.call("sum", xs), [1n, 2n, 3n, 2n], 8.0],
    [M.call("min", xs), [1n, 2n, 3n, 2n], 1.0], [M.call("max", xs), [1n, 2n, 3n, 2n], 3.0],
    [M.call("numel", M.call("unique", xs)), [1n, 2n, 3n, 2n], 3.0],
    [M.arrayfun("p", xs, M.binary(".*", p, 2n)), [1n, 2n, 3n, 2n], [2.0, 4.0, 6.0, 4.0]],
    [M.arrayfun("p", xs, M.binary(">", p, 1n)), [1n, 2n, 3n, 2n], [false, true, true, true]],
    [M.arrayfun("p", xs, p), [true, 2n], [1.0, 2.0]], [M.call("all", xs), [], true], [M.call("any", xs), [], false],
    [M.call("sum", xs), [], 0.0], [M.call("min", xs), [], []], [M.call("max", xs), [NaN, 2n], 2.0], [M.call("any", xs), [NaN], false],
    [M.call("all", xs), [NaN], true], [M.call("nnz", xs), [NaN, 0n], 1.0], [M.call("min", xs), [true, false], false],
    [M.call("sum", xs), [true, 2n], 3.0], [M.call("unique", xs), ["b", "a", "b"], ["a", "b"]], [M.call("unique", xs), [true, true], [true]],
    [M.call("unique", xs), [3n, 1n, 3n], [1.0, 3.0]], [M.call("numel", xs), 5n, 1.0], [M.index(xs, 1n), 5n, 5.0],
    [M.arrayfun("p", xs, M.binary("+", p, 1n)), 5n, [6.0]], [M.call("ismember", 1n, xs), [true], true], [M.call("numel", xs), "abc", 1.0],
    [M.call("numel", xs), new Map([["a", 1n]]), 1.0],
  ];
  const ERRORS: [any, unknown, Function, string][] = [
    [M.index(xs, 0n), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, 1.5), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, "a"), [1n, 2n], ValueError, "Array indices must be positive integers or logical values."],
    [M.index(xs, 5n), [1n, 2n, 3n, 2n], ValueError, "Index exceeds the number of array elements. Index must not exceed 4."],
    [M.arrayfun("p", xs, "a"), [1n, 2n], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."],
    [M.arrayfun("p", xs, M.binary("+", xs, 0n)), [1n, 2n], TypeError, "Operator '+' is not supported for operands of type 'array' and 'double'."],
    [M.arrayfun("p", xs, xs), [1n, 2n], ValueError, "Non-scalar in Uniform output, at index 1, output 1. Set 'UniformOutput' to false."],
    [M.call("sum", xs), ["a"], TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("all", xs), new Map([["a", 1n]]), TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("unique", xs), [new Map([["a", 1n]])], TypeError, "Invalid data type. First argument must be numeric or logical."],
    [M.call("ismember", xs, xs), [1n], TypeError, "ismember takes a scalar and an array of scalars."],
    [M.binary("==", xs, 1n), [1n], TypeError, "Operator '==' is not supported for operands of type 'array' and 'double'."],
    [M.binary("<", xs, 1n), new Map([["a", 1n]]), TypeError, "Operator '<' is not supported for operands of type 'struct' and 'double'."],
  ];
  for (const [expression, value, expected] of VALUES) {
    const result = MV.OfAny(mat(expression), { xs: value });
    assert(alike(result, expected), `${MText.ToText(mat(expression as any))}: ${String(result)}`);
  }
  for (const [expression, value, error, message] of ERRORS) raises(error, () => MV.OfAny(mat(expression), { xs: value }), message);
  assert(Number.isNaN(MV.OfAny(mat(M.call("max", xs)), { xs: [NaN] })) && alike(MV.OfAny(mat(M.call("unique", xs)), { xs: [NaN, 1n, NaN] }), [1.0, NaN, NaN]));
  assert(alike(MV.OfAny(mat(M.field(xs, "ranks")), { xs: squad }), [1.0, 2.0, 3.0])); // a list property is an array of doubles
  assert(MV.OfAny(mat(M.call("sum", M.field(xs, "ranks"))), { xs: squad }) === 6.0);
  raises(ValueError, () => MV.OfAny(mat(M.arrayfun("p", null, 1n))), "an arrayfun needs an array");
}

<a id="DIA-17"></a>
## DIA-17 · Excel arrays: MAP and LAMBDA, AND and OR of any number of arguments, and the functions of arrays

In [ ]:
{
  const [xs, p] = [X.name("xs"), X.name("p")];
  assert(XText.ToText(xl(X.function("AND", X.map_("p", X.field(xs, "pins"), X.infix(">", p, 0n))))) === "=AND(MAP(xs.pins, LAMBDA(p, p > 0)))");
  assert(XText.ToText(xl(X.function("INDEX", xs, X.infix("+", p, 1n)))) === "=INDEX(xs, p + 1)");
  assert(X.DIALECT.validate(xl(X.map_("p", xs, p)), ["xs"]).length === 0
    && same([...SY.free(xl(X.map_("p", xs, X.infix("+", p, X.name("q")))), X.DIALECT)].sort(), ["q", "xs"]));
  assert(X.DIALECT.validate(xl(X.function("AND", 1n, 2n, 3n))).length === 0 && same(X.DIALECT.validate(xl(X.function("ROWS", xs, 1n)), ["xs"]), ["ROWS takes 1 arguments, got 2"]));
  assert(X.DIALECT.infer(xl(X.function("AND", xs)), { xs: D.Anything }).name() === "logical");
  assert(X.DIALECT.infer(xl(X.map_("p", xs, p)), { xs: D.Anything }) === D.Anything);
  assert(XV.OfAny(xl(X.function("AND", true, true, false))) === false && XV.OfAny(xl(X.function("OR", false, false, true))) === true);
  // IF takes 2 or 3 arguments, and AND, OR, SUM, MIN and MAX 1 to 255, as Excel's do.
  assert(X.DIALECT.validate(xl(X.function("IF", true, 1n))).length === 0 && XV.OfAny(xl(X.function("IF", false, 1n))) === false && XV.OfAny(xl(X.function("IF", true, 1n))) === 1.0);
  assert(same(X.DIALECT.validate(xl(X.function("IF", true))), ["IF takes 2 or 3 arguments, got 1"]) && same(X.DIALECT.validate(xl(X.function("SUM"))), ["SUM takes 1 to 255 arguments, got 0"]));
  assert(XV.OfAny(xl(X.function("SUM", 1n, xs, true)), { xs: [2n, "a", true] }) === 4.0 && XV.OfAny(xl(X.function("MAX", xs, 7n)), { xs: [1n, 2n] }) === 7.0);
  assert(XV.OfAny(xl(X.function("MIN", 3n, X.name("e"))), { e: XD.Error.of("#N/A") }) === XD.Error.of("#N/A"));
  const arities: [unknown, string][] = [[X.function("IF", true, 1n, 2n, 3n), "IF takes 2 or 3 arguments, got 4"],
    [X.function("AND"), "AND takes 1 to 255 arguments, got 0"], [X.function("MIN"), "MIN takes 1 to 255 arguments, got 0"]];
  for (const [expression, message] of arities) raises(TypeError, () => XV.OfAny(xl(expression)), message);
  // VSTACK stacks its arguments into one array, an array's elements in turn, errors among them.
  assert(XText.ToText(xl(X.function("VSTACK", 1n, xs))) === "=VSTACK(1, xs)"
    && same(X.DIALECT.validate(xl(X.function("VSTACK")), ["xs"]), ["VSTACK takes 1 to 254 arguments, got 0"]));
  const stacked = XV.OfAny(xl(X.function("VSTACK", 1n, xs, X.name("e"))), { xs: [2n, "a"], e: XD.Error.of("#N/A") }) as unknown[];
  assert(equal(stacked.slice(0, 3), [1, 2, "a"]) && stacked[3] === XD.Error.of("#N/A"));
  raises(TypeError, () => XV.OfAny(xl(X.function("VSTACK"))), "VSTACK takes 1 to 254 arguments, got 0");
  const VALUES: [any, unknown, unknown][] = [
    [X.function("AND", X.map_("p", xs, X.infix(">", p, 0n))), [1n, 2n, 3n, 2n], true],
    [X.function("OR", X.map_("p", xs, X.infix(">", p, 2n))), [1n, 2n, 3n, 2n], true],
    [X.function("SUM", X.map_("p", xs, X.function("IF", X.infix(">=", p, 2n), 1n, 0n))), [1n, 2n, 3n, 2n], 3.0],
    [X.function("ROWS", xs), [1n, 2n, 3n, 2n], 4.0], [X.function("INDEX", xs, 2n), [1n, 2n, 3n, 2n], 2.0],
    [X.function("INDEX", xs, 2.9), [1n, 2n, 3n, 2n], 2.0], [X.function("INDEX", xs, 5n), [1n, 2n], XD.Error.of("#REF!")],
    [X.function("INDEX", xs, 0n), [1n, 2n], XD.Error.of("#REF!")], [X.function("INDEX", xs, "x"), [1n, 2n], XD.Error.of("#VALUE!")],
    [X.function("MATCH", 3n, xs, 0n), [1n, 2n, 3n, 2n], 3.0], [X.function("MATCH", "B", xs, 0n), ["a", "b"], 2.0],
    [X.function("MATCH", 5n, xs, 0n), [1n, 2n], XD.Error.of("#N/A")], [X.function("MATCH", "1", xs, 0n), [1n, 2n], XD.Error.of("#N/A")],
    [X.function("MATCH", 1n, xs, 1n), [1n, 2n], XD.Error.of("#VALUE!")], [X.function("MATCH", true, xs, 0n), [1n, true], 2.0],
    [X.function("ISNUMBER", X.function("MATCH", 2n, xs, 0n)), [1n, 2n], true],
    [X.function("ISNUMBER", X.function("MATCH", 7n, xs, 0n)), [1n, 2n], false], [X.function("ISNUMBER", "1"), [], false],
    [X.function("SUM", xs), [1n, 2n, 3n, 2n], 8.0], [X.function("SUM", xs), [1n, "a", true, 2n], 3.0],
    [X.function("SUM", xs), [1n, XD.Error.of("#N/A")], XD.Error.of("#N/A")], [X.function("SUM", true), [], 1.0],
    [X.function("SUM", "2"), [], 2.0], [X.function("SUM", "a"), [], XD.Error.of("#VALUE!")], [X.function("MIN", xs), [3n, 1n, 2n], 1.0],
    [X.function("MAX", xs), [3n, 1n, 2n], 3.0], [X.function("MAX", xs), ["a"], 0.0], [X.function("MIN", xs), [], 0.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), [1n, 2n, 3n, 2n], 3.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), ["a", "A", 1n, true, "1"], 4.0],
    [X.function("ROWS", X.function("UNIQUE", xs)), [XD.Error.of("#N/A"), XD.Error.of("#N/A"), new Map([["a", 1n]]), new Map([["a", 1n]])], 3.0],
    [X.function("ROWS", xs), 5n, 1.0], [X.function("INDEX", xs, 1n), 5n, 5.0],
    [X.function("ROWS", X.map_("p", xs, X.infix("*", p, 2n))), 5n, 1.0], [X.function("AND", xs), [true, "a", 1n], true],
    [X.function("AND", xs), ["a"], XD.Error.of("#VALUE!")], [X.function("OR", xs), [], XD.Error.of("#VALUE!")],
    [X.function("AND", xs, false), [true], false], [X.function("OR", xs), [0n, XD.Error.of("#DIV/0!")], XD.Error.of("#DIV/0!")],
    [X.function("INDEX", X.map_("p", xs, X.infix("-", p, "a")), 1n), [1n], XD.Error.of("#VALUE!")],
    [X.function("ROWS", X.map_("p", xs, p)), XD.Error.of("#REF!"), XD.Error.of("#REF!")], [X.infix("+", xs, 1n), [1n], XD.Error.of("#VALUE!")],
    [X.infix("=", xs, 1n), [1n], XD.Error.of("#VALUE!")],
    [X.function("MATCH", 1n, xs, "a"), [1n], XD.Error.of("#VALUE!")], [X.function("UNIQUE", xs), XD.Error.of("#REF!"), XD.Error.of("#REF!")],
  ];
  assert(XV.OfAny(xl(X.function("INDEX", xs, X.name("n"))), { xs: [1n], n: Infinity }) === XD.Error.of("#NUM!"));
  for (const [expression, value, expected] of VALUES) {
    const result = XV.OfAny(xl(expression), { xs: value });
    assert(result === expected && typeof result === typeof expected, `${XText.ToText(xl(expression as any))}: ${String(result)}`);
  }
  assert(same(XV.OfAny(xl(X.field(xs, "ranks")), { xs: squad }) as unknown[], [1.0, 2.0, 3.0])); // a list property is an array of numbers
  assert(XV.OfAny(xl(X.function("SUM", X.field(xs, "ranks"))), { xs: squad }) === 6.0);
  assert(same(XV.OfAny(xl(X.map_("p", xs, X.infix(">", p, 1n))), { xs: [1n, 2n] }) as unknown[], [false, true]));
}

<a id="DIA-18"></a>
## DIA-18 · A literal is a domain representation: printers write its spelling where it has one, else a default spelling; its value is what evaluates

In [ ]:
{
  /** A literal of `dialect` holding `value`, spelled `text`, as a parser would write it. */
  const spelled = (dialect: F.Dialect, value: unknown, text: string) => term((u: any) => u.literal(
    (r: any) => (value !== null ? r.value((v: any) => v.as_scalar(value)) : r).text(text)), dialect);
  for (const [dialect, printer, text, value, written] of [
    [P.DIALECT, PText.ToText, "0x10", 16n, "16"],
    [M.DIALECT, MText.ToText, "1e1", 10.0, "10.0"],
    [X.DIALECT, XText.ToText, "1E+1", 10.0, "=10.0"],
    [Latex.Expressions.DIALECT, LText.ToText, "10.00", 10.0, "10.0"],
    [SystemVerilog.Expressions.DIALECT, SVText.ToText, "'d16", 16n, "16"],
    [TS.DIALECT, TSText.ToText, "0x10", 16.0, "16"],
  ] as [F.Dialect, (expression: unknown) => string, string, unknown, string][]) {
    const literal = spelled(dialect, value, text);
    assert(printer(literal) === (dialect === X.DIALECT ? "=" + text : text) && dialect.validate(literal).length === 0);
    assert(printer(term(value, dialect)) === written); // no spelling: the value's default one
  }
  assert(PText.ToText(term(P.binop("**", spelled(P.DIALECT, -2n, "-2"), 2n), P.DIALECT)) === "(-2) ** 2"); // a spelling's precedence
  assert(PText.ToText(term(P.unaryop("-", spelled(P.DIALECT, -1n, "-1")), P.DIALECT)) === "--1"); // as its value's would be
  assert(MText.ToText(term(M.unary("-", spelled(M.DIALECT, -1n, "-1")), M.DIALECT)) === "--1");
  const LX = Latex.Expressions;
  assert(LText.ToText(term(LX.binary("+", 1n, spelled(LX.DIALECT, -1n, "-1")), LX.DIALECT)) === "1 + -1");
  assert(PV.OfAny(spelled(P.DIALECT, 16n, "0x10")) === 16n && MV.OfAny(spelled(M.DIALECT, 10.0, "1e1")) === 10.0); // its value
  const invalid = spelled(P.DIALECT, null, "08");
  assert(PText.ToText(invalid) === "08" && same(P.DIALECT.validate(invalid), ["a literal needs a value"])); // kept as written
}

<a id="DIA-19"></a>
## DIA-19 · Python3's kinds beyond the narrower ones: literals keeping what their text encodes, f-strings as written, each kind's own checks and what it binds, stored as any term

In [ ]:
{
  const checked = (spec: unknown, bound: string[] = []): string[] => P.DIALECT.validate(py(spec), bound); // its problems
  const shown = (spec: unknown): string => PText.ToText(py(spec)); // its source
  const [x, y] = [P.name("x"), P.name("y")];
  // A Python literal keeps what its text encodes beside its text, its problems and its value: a string's prefix and
  // quotes, an int's base, digit separators; Python3's own constant holds None, ... and complex numbers. (Read from
  // source by the Python implementation's `FromText`; written here through the builders.)
  assert(shown(P.constant(16n, "0x10")) === "0x10" && equal(F.form(py(P.constant(null, "None"))).attributes,
    new Map([["singleton", "None"], ["text", "None"]]))); // written with its spelling
  assert(shown(P.joinedstr(["a", P.formattedvalue(x, null, null, true), P.formattedvalue(P.set_(1n))])) === "f'a{x=}{ {1}}'"); // unspelled
  assert(shown(P.joinedstr([P.formattedvalue(x, null, null, false, "x = ")])) === "f'{x = }'"); // a field's expression as written
  assert(shown(P.templatestr([P.interpolation(x, "s", ["0", P.formattedvalue(y)])], "t", "\"")) === "t\"{x!s:0{y}}\""); // a format spec's fields are formatted values, as `ast` has them
  for (const [spec, problems] of [ // each new kind's own checks
    [(u: any) => u[P.CONSTANT]((r: any) => r), ["a constant needs a singleton or a complex number"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.singleton("None").imag(1.0)), ["a constant holds a singleton or a complex number, not both"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.singleton("Ellipsis")), ["a constant's singleton must be None or ..., got 'Ellipsis'"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.real(1.0)), ["a complex constant needs an imag"]],
    [(u: any) => u[P.CONSTANT]((r: any) => r.text("1__0j").problems(["invalid decimal literal"])), ["invalid decimal literal"]],
    [(u: any) => u[P.LITERAL]((r: any) => r.value((v: any) => v.as_scalar("x")).prefix("q").quote("`").base(3n)),
      ["a literal's prefix cannot be 'q'", "a literal's quote cannot be '`'", "a literal's base cannot be 3"]],
    [P.joinedstr(["x"], "b"), ["a joinedstr's prefix cannot be 'b'"]],
    [P.templatestr(["x"], "Rt", "`"), ["a templatestr's quote cannot be '`'"]],
    [P.joinedstr([P.formattedvalue(1n, "q", ["0", P.formattedvalue(2n, "z")])]),
      ["a formattedvalue's conversion cannot be 'q'", "a formattedvalue's conversion cannot be 'z'"]], // its spec's fields too
    [P.concatenatedstring("a"), ["a concatenatedstring needs two strings"]],
    [P.set_(), ["a set needs an element"]],
    [P.boolop("and", 1n), ["a boolop needs two values"]],
    [P.listcomp(1n), ["a listcomp needs a for clause"]],
    [P.call(P.name("f"), P.keyword("a b", 1n)), ["a keyword's arg must be an identifier, got 'a b'"]],
    [(u: any) => u[P.COMPARE]((r: any) => r.left(P.constant(1n)).comparisons([
      (a: any) => a.as_object((o: any) => o.comparator(P.constant(2n)))])), ["a comparison needs an op"]],
    [P.namedexpr(P.attribute(P.name("o"), "a"), 1n), ["a namedexpr's target must be a name"]],
    [P.lambda_(1n, [P.arg("a", null, P.name("int"))]), ["a lambda's parameters have no annotations"]],
    [P.lambda_(1n, ["a b"]), ["a lambda's parameter must be an identifier, got 'a b'"]],
    [(u: any) => u[P.LAMBDA]((r: any) => r.body(P.constant(1n)).args((a: any) => a.args([
      (e: any) => e.as_object((o: any) => o.default_value(P.constant(1n)))]))), ["a lambda's parameter needs an arg"]],
    [(u: any) => u[P.LISTCOMP]((r: any) => r.elt(P.constant(1n)).generators([
      (a: any) => a.as_object((o: any) => o.iter(P.name("xs")))])), ["a listcomp needs a generators 0 target"]],
    [P.lambda_(1n), []], [P.yield_(), []], [P.dictcomp(1n, null, P.comprehension("x", P.name("xs"))), []], // optional slots
    [P.lambda_(P.binop("+", P.name("a"), P.name("k")), ["a"], { vararg: "v", kwonlyargs: ["k"], kwarg: "w" }), []],
  ] as [unknown, string[]][]) {
    assert(same(checked(spec, ["f", "xs", "o"]), problems), Repr.repr(checked(spec, ["f", "xs", "o"])));
  }
  // What each new kind binds: a comprehension its targets, a lambda its parameters, a named expression its target alone
  for (const [spec, free] of [
    [P.listcomp(P.name("q"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("q"))), P.name("xs"), [P.name("p")])), ["xs"]],
    [P.listcomp(1n, P.comprehension(P.attribute(P.name("o"), "a"), P.name("xs"))), ["o", "xs"]], // an attribute binds nothing
    [P.dictcomp(P.name("k"), P.name("v"), P.comprehension(P.parenthesized(P.list_(P.name("k"), P.name("v"))), P.name("d"))), ["d"]],
    [P.setcomp(P.name("q"), P.comprehension("p", P.name("ps")), P.comprehension("q", P.name("p"))), ["ps"]], // later clauses see earlier
    [P.lambda_(P.binop("+", P.name("a"), P.name("b")), ["a", P.arg("b", P.name("a"))]), ["a"]], // a default: the scope's
    [P.boolop("and", P.namedexpr("y", 1n), P.name("y")), ["y"]], // beyond itself, the caller binds a named expression's target
    [(u: any) => u[P.LISTCOMP]((r: any) => r.elt(P.name("e")).generators([
      (a: any) => a.as_object((o: any) => o.iter(P.name("xs")))])), ["e", "xs"]], // no target: nothing bound
  ] as [unknown, string[]][]) {
    assert(same([...SY.free(py(spec), P.DIALECT)].sort(), free), Repr.repr([...SY.free(py(spec), P.DIALECT)]));
  }
  // Printed as held, what the writers would not write: a complex number of no real part, a parameter of no name
  assert(shown((u: any) => u[P.CONSTANT]((r: any) => r.imag(2.0))) === "2j");
  assert(shown((u: any) => u[P.LAMBDA]((r: any) => r.body(P.constant(1n)).args((a: any) => a.args([
    (e: any) => e.as_object((o: any) => o.default_value(P.constant(1n)))])))) === "lambda =1: 1");
  const spelled = py((u: any) => u[P.LITERAL]((r: any) => r.value((v: any) => v.as_scalar(255n)).text("0x_FF").base(16n).separators(true)));
  assert(equal(F.form(spelled).attributes, new Map<string, unknown>([["value", 255n], ["text", "0x_FF"], ["base", 16n], ["separators", true]]))
    && PText.ToText(spelled) === "0x_FF");
  // A union of helpers holding none (an f-string's value with no branch written) holds nothing, read or printed
  const empty = py((u: any) => u[P.JOINEDSTR]((r: any) => r.values([(a: any) => a.as_union((v: any) => v)])));
  assert(!F.form(empty).attributes.has("values") && PText.ToText(empty) === "f''");
  // Each new kind is stored as any term, its helpers nested in it: every kind of FRM's sample round-trips; here a few
  for (const spec of [P.lambda_(1n, [P.arg("a", 2n)], { posonlyargs: ["p"] }), P.joinedstr(["a", P.formattedvalue(x, "r", ["0"])]),
    P.call("f", P.keyword(null, x)), P.compare("<", 1n, 2n, ["in", x])]) {
    assert(roundTrips(py(spec)) && F.same(py(P.DIALECT.make(F.form(py(spec)))), py(spec)));
  }
}

<a id="DIA-20"></a>
## DIA-20 · Python3 is complete: each node of `ast` an expression is or holds, 3.0 to 3.15, kept as data, is a kind, a helper or an operator of a kind, or is listed as not an expression with its reason

In [ ]:
import { readFileSync as readGrammar } from "node:fs";

{
  interface GrammarNode {
    node: string; class: string; since: string; until?: string; kinds?: string[]; helper?: string; op?: string;
    kind?: string; not?: string;
  }
  const grammar = globalThis.JSON.parse(readGrammar("../../grammars/python3-ast.json", "utf8")) as {
    nodes: GrammarNode[]; concrete: { kind: string }[] }; // `ast`'s nodes, as data
  const kinds = P.DIALECT.kinds();
  const mapped = new Set<string>();
  for (const node of grammar.nodes) { // each a kind, a helper, an operator of a kind, or not an expression, with its reason
    const ways = (["kinds", "helper", "op", "not"] as const).filter((way) => way in node);
    assert(ways.length === 1, node.node);
    if (node.kinds !== undefined) {
      assert(node.kinds.length > 0 && node.kinds.every((kind) => kinds.has(kind)), node.node);
      node.kinds.forEach((kind) => mapped.add(kind));
    } else if (node.helper !== undefined) { // a value object a kind holds, not a term
      const helper = store.registered(`Expressions.Python3.${node.helper}`);
      assert(helper !== null && helper !== undefined && !helper.ref && !kinds.has(node.helper.toLowerCase()), node.node);
    } else if (node.op !== undefined) {
      assert(kinds.get(node.kind as string)!.vocabulary!.has(node.op), node.node);
    } else {
      assert(typeof node.not === "string" && node.not !== "", node.node);
    }
  }
  const concrete = grammar.concrete.map((entry) => entry.kind); // what `ast` does not keep: parentheses
  assert(same([...kinds.keys()].sort(), [...new Set([...mapped, ...concrete, ...P.EXTENSIONS, "error", "missing"])].sort())); // every kind is `ast`'s, but these
  for (const kind of ["boolop", "binop", "unaryop", "compare"]) { // every operator of the vocabularies is `ast`'s, and the other way
    assert(same([...kinds.get(kind)!.vocabulary!.keys()].sort(),
      grammar.nodes.filter((node) => node.kind === kind).map((node) => node.op as string).sort()), kind);
  }
  // The running Python's `ast` against the data is the Python implementation's alone: `ast` is Python's.
}

<a id="DIA-21"></a>
## DIA-21 · TypeScript5: typescript-estree's expressions in mbse-programs' shapes, its types and patterns each a union of its own; literals keeping what their text encodes; each kind's checks and bindings; evaluation of literals alone

In [ ]:
{
  const ts = (spec: unknown): any => term(spec, TS.DIALECT); // TypeScript5 has no writers: its terms, through the builders
  const tsChecked = (spec: unknown, bound: string[] = []): string[] => TS.DIALECT.validate(ts(spec), bound); // its problems
  // Its unions: its terms, and beside them its sublanguages, each a union of its own; a schema is a branch of every
  // union whose positions hold it ("(b) as you recommended"), named alike in each.
  const unions = new Map(["Term", "Pattern", "Type"].map((name) => [name, store.registered(`Expressions.TypeScript5.${name}`)]));
  const branches = new Map([...unions].map(([name, union]) =>
    [name, new Map<string, any>(union.branches.map((b: any) => [b.name, b.schema]))]));
  const tags = (name: string): string[] => [...branches.get(name)!.keys()];
  assert(same(tags("Term"), [...TS.DIALECT.kinds().keys()]) && tags("Term").length === 46);
  assert(same(tags("Pattern"), ["arraypattern", "objectpattern", "assignmentpattern", "restelement", "identifier",
    "memberexpression", "parenthesizedexpression", "tsasexpression", "tssatisfiesexpression", "tstypeassertion",
    "tsnonnullexpression", "property", "elision", "tsparameterproperty"]));
  assert(tags("Type").filter((tag) => tag.startsWith("ts")).length === 37); // mbse-programs' 37: typescript-estree's 36, and parentheses
  assert(same(tags("Type").slice(37), ["identifier", "thisexpression", "literal", "unaryexpression", "templateliteral",
    "objectexpression"])); // the terms a type holds
  assert(branches.get("Pattern")!.get("identifier") === branches.get("Term")!.get("identifier")
    && branches.get("Type")!.get("identifier") === branches.get("Term")!.get("identifier"));
  assert(branches.get("Term")!.get("arrayexpression") === store.registered("Expressions.Framework.List")); // the shape matches exactly
  assert(branches.get("Term")!.get("literal").refines === store.registered("Expressions.Framework.Literal"));
  assert(store.registered("Expressions.TypeScript5.Expression").properties.get("root").schema === unions.get("Term"));
  assert(branches.get("Term")!.get("assignmentexpression").properties.get("left").schema === unions.get("Pattern")); // what holds a pattern
  assert(branches.get("Term")!.get("tsasexpression").properties.get("typeAnnotation").schema === unions.get("Type")); // and a type
  assert(store.registered("Expressions.TypeScript5.TSQualifiedName").properties.get("right").schema === branches.get("Term")!.get("identifier"));
  // A sublanguage's value, as any helper, is read as a part: x = ([a, b = 1]: number[]) => a as string
  const arrow = ts((u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.identifier((i: any) => i.name("x"))).operator("=").right(
    (v: any) => v.arrowfunctionexpression((a: any) => a.params([(p: any) => p.arraypattern((ap: any) => ap.elements([
      (e: any) => e.identifier((i: any) => i.name("a")),
      (e: any) => e.assignmentpattern((s: any) => s.left((t: any) => t.identifier((i: any) => i.name("b"))).right(
        (t: any) => t.literal((li: any) => li.value((n: any) => n.as_scalar(1.0)))))]).typeAnnotation(
      (ta: any) => ta.typeAnnotation((t: any) => t.tsarraytype((at: any) => at.elementType((y: any) => y.tsnumberkeyword((k: any) => k))))))])
      .body((b: any) => b.tsasexpression((s: any) => s.expression((e: any) => e.identifier((i: any) => i.name("a"))).typeAnnotation(
        (t: any) => t.tsstringkeyword((k: any) => k))))))));
  const form = F.form(arrow);
  const left = form.attributes.get("left") as F.Part;
  assert(form.kind === "assignmentexpression" && left.kind === "identifier" && equal(left.attributes, map({ name: "x" })));
  const params = F.form(form.arguments[0]).attributes.get("params") as F.Part[];
  const first = params[0]!;
  assert(same(params.map((p) => p.kind), ["arraypattern"]) && same((first.attributes.get("elements") as F.Part[]).map((p) => p.kind),
    ["identifier", "assignmentpattern"]) && first.size === 1); // the default, a term, is an argument
  assert(((first.attributes.get("typeAnnotation") as F.Part).attributes.get("typeAnnotation") as F.Part).kind === "tsarraytype");
  const node = TS.tree(form.arguments[0]); // its helpers nested, as the printer and the checks read it
  assert(node.kind === "arrowfunctionexpression" && F.kindOf(node.slots.get("body"))[0] === "tsasexpression");
  const pattern = (node.helpers.get("params") as TypeScript5.Expressions.Node[])[0]!;
  const elements = pattern.helpers.get("elements") as TypeScript5.Expressions.Node[];
  assert(pattern.kind === "arraypattern" && equal((elements[1]!.helpers.get("left") as TypeScript5.Expressions.Node).attributes, map({ name: "b" })));
  assert(equal(F.form(elements[1]!.slots.get("right")).attributes, map({ value: 1.0 })) && node.many.length === 0);
  assert(same([...TS.bindingNames(pattern)], ["a", "b"]) && TS.bindingNames(null).size === 0);
  assert(TS.DIALECT.validate(arrow).length === 0 && roundTrips(arrow) && F.same(ts(TS.DIALECT.make(form)), arrow));
  assert(SY.free(arrow, TS.DIALECT).size === 0); // its parameters bind its body
  // A literal keeps what its text encodes beside its text, its problems and its value: a number is a float, a bigint an
  // int; TypeScript5's own constant holds null and a regular expression, which no schema holds.
  for (const [spec, attributes, domain] of [
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(255.0)).text("0x_FF").base(16n).separators(true)),
      { value: 255.0, text: "0x_FF", base: 16n, separators: true }, "number"],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(1000n)).text("1_000n").separators(true)),
      { value: 1000n, text: "1_000n", separators: true }, "bigint"],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("a")).text("'a'").quote("'")),
      { value: "a", text: "'a'", quote: "'" }, "string"],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(true)).text("true")), { value: true, text: "true" }, "boolean"],
  ] as [unknown, Record<string, unknown>, string][]) {
    const spelled = ts(spec);
    assert(equal(F.form(spelled).attributes, map(attributes)) && TS.DIALECT.validate(spelled).length === 0);
    assert(TS.DIALECT.infer(spelled).name() === domain && TV.OfAny(spelled) === attributes.value); // its value
  }
  const nil = ts((u: any) => u.constant((r: any) => r.singleton("null").text("null")));
  const regex = ts((u: any) => u.constant((r: any) => r.pattern("a+").flags("gu")));
  assert(equal(F.form(regex).attributes, map({ pattern: "a+", flags: "gu" })) && TS.DIALECT.validate(regex).length === 0
    && TS.DIALECT.validate(nil).length === 0);
  assert(TS.DIALECT.infer(nil).name() === "null" && TS.DIALECT.infer(regex).name() === "RegExp");
  assert(same(TS.DIALECT.kinds().get("literal")!.spelling, ["text", "problems", "quote", "base", "separators"]));
  raises(TypeError, () => TS.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'"); // no literal holds bytes
  assert(equal(F.form(ts(1n)).attributes, map({ value: 1n })) && equal(F.form(ts(1.5)).attributes, map({ value: 1.5 }))); // a bigint, a number
  // Evaluation is shallow: a literal's value; every other kind refused by name, a constant among them
  for (const [spec, message] of [[(u: any) => u.thisexpression((r: any) => r), "TypeScript5 thisexpression is not evaluated"],
    [(u: any) => u.constant((r: any) => r.singleton("null")), "TypeScript5 constant is not evaluated"]] as [unknown, string][]) {
    raises(NotImplementedError, () => TV.OfAny(ts(spec)), message);
  }
  raises(NotImplementedError, () => TV.OfAny(arrow), "TypeScript5 assignmentexpression is not evaluated");
  assert(same([...TV.EVALUATED], ["literal"]));
  // Each kind's own checks, mbse-programs', and those of the types, patterns and JSX it holds
  const x = (e: any) => e.identifier((i: any) => i.name("x")); // a spec of `x`, in any union that has identifiers
  for (const [spec, problems] of [
    [(u: any) => u.identifier((r: any) => r.name("a b")), ["an identifier's name cannot be 'a b'"]],
    [(u: any) => u.identifier((r: any) => r.name("$a")), []], // `$` is a letter
    [(u: any) => u.privateidentifier((r: any) => r.name("1")), ["a privateidentifier's name cannot be '1'"]],
    [(u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("s")).quote("`").base(3n)),
      ["a literal's quote cannot be '`'", "a literal's base cannot be 3"]],
    [(u: any) => u.literal((r: any) => r.text("08n").problems(["a bigint has no leading zeros"])), ["a bigint has no leading zeros"]],
    [(u: any) => u.constant((r: any) => r), ["a constant needs a singleton or a regular expression"]],
    [(u: any) => u.constant((r: any) => r.text("/a").problems(["unterminated regular expression"])), ["unterminated regular expression"]],
    [(u: any) => u.constant((r: any) => r.singleton("null").pattern("a")), ["a constant holds a singleton or a regular expression, not both"]],
    [(u: any) => u.constant((r: any) => r.singleton("undefined")), ["a constant's singleton must be null, got 'undefined'"]],
    [(u: any) => u.constant((r: any) => r.flags("g")), ["a regular expression needs a pattern"]],
    [(u: any) => u.constant((r: any) => r.pattern("a")), []], // its flags, none
    [(u: any) => u.constant((r: any) => r.pattern("a").flags("gq")), ["a regular expression's flags cannot be 'gq'"]],
    [(u: any) => u.constant((r: any) => r.pattern("a").flags("gg")), ["a regular expression's flags cannot be 'gg'"]],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.raw("a")]).expressions([x])),
      ["a templateliteral needs one more quasi than expressions"]],
    [(u: any) => u.sequenceexpression((r: any) => r.expressions([x])), ["a sequenceexpression needs two expressions"]],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((y: any) => y.types([
      (z: any) => z.tsintersectiontype((w: any) => w.types([(v: any) => v.tsanykeyword((k: any) => k)]))])))),
      ["a tsuniontype needs two types", "a tsintersectiontype needs two types"]], // a type's, within it
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstemplateliteraltype((y: any) => y.quasis([
      (q: any) => q.raw("a")]).types([(v: any) => v.tsstringkeyword((k: any) => k)])))),
      ["a tstemplateliteraltype needs one more quasi than types"]],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.typeParameters((d: any) => d.params([]))), ["a tstypeparameterdeclaration needs a type parameter"]],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.identifier((i: any) => i.name("a b"))])),
      ["an identifier's name cannot be 'a b'"]], // a pattern's, within it
    [(u: any) => u.binaryexpression((r: any) => r.left(x).operator("<>").right(x)), []], // an operator outside its vocabulary
    [(u: any) => u.binaryexpression((r: any) => r.left(x).right(x)), ["a binaryexpression needs an operator"]],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x)), ["a tsasexpression needs a typeAnnotation"]], // a helper it needs
    [(u: any) => u.assignmentexpression((r: any) => r.right(x)), ["an assignmentexpression needs a left", "an assignmentexpression needs an operator"]],
    [(u: any) => u.unaryexpression((r: any) => r.operator("-")), ["an unaryexpression needs an argument"]],
    [(u: any) => u.yieldexpression((r: any) => r), []], [(u: any) => u.importexpression((r: any) => r.source(x)), []], // optional slots
    [(u: any) => u.arrowfunctionexpression((r: any) => r), []], // a body of statements, not part of the term
    [(u: any) => u.classexpression((r: any) => r), []],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.assignmentpattern((a: any) => a.left(x))).operator("=").right(x)),
      ["an assignmentexpression needs a left right"]], // a pattern's term, labelled by its path
  ] as [unknown, string[]][]) {
    const found = tsChecked(spec, ["x", "$a"]);
    assert(same(found, problems), `${Repr.repr(found)} != ${Repr.repr(problems)}`);
  }
  assert(same(TS.DIALECT.validate(ts((u: any) => u.binaryexpression((r: any) => r.left(x).operator("<>").right(x))), ["x"], true),
    ["'<>' is not a core operation"]));
  // What each kind binds: a function's and an arrow's parameters; a member's property and a key, not computed, are names
  // rather than references
  for (const [spec, free] of [
    [(u: any) => u.memberexpression((r: any) => r.object(x).property((p: any) => p.identifier((i: any) => i.name("y")))), ["x"]],
    [(u: any) => u.memberexpression((r: any) => r.object(x).computed(true).property((p: any) => p.identifier((i: any) => i.name("y")))), ["x", "y"]],
    [(u: any) => u.objectexpression((r: any) => r.properties([(p: any) => p.property((q: any) => q.key((k: any) => k.identifier((i: any) => i.name("k")))
      .value((v: any) => v.literal((li: any) => li.value((n: any) => n.as_scalar("s"))))), (p: any) => p.property((q: any) => q.computed(true)
      .key(x).value(x)), (p: any) => p.property((q: any) => q.key((k: any) => k.literal((li: any) => li.value((n: any) => n.as_scalar("s")))).value(x)),
      (p: any) => p.property((q: any) => q.value(x))])), ["x"]], // a key that is a literal, or none, names nothing
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstypeliteral((y: any) => y.members([
      (m: any) => m.tspropertysignature((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("k")))),
      (m: any) => m.tsmethodsignature((s: any) => s.computed(true).key((k: any) => k.identifier((i: any) => i.name("z")))),
      (m: any) => m.comment((c: any) => c.text(" c"))])))), ["x", "z"]], // a type literal's keys, within a type
    [(u: any) => u.functionexpression((r: any) => r.params([(p: any) => p.objectpattern((o: any) => o.properties([
      (q: any) => q.property((s: any) => s.key((k: any) => k.identifier((i: any) => i.name("a"))).value((v: any) => v.identifier((i: any) => i.name("b")))),
      (q: any) => q.restelement((s: any) => s.argument((a: any) => a.identifier((i: any) => i.name("c"))))])),
    (p: any) => p.tsparameterproperty((s: any) => s.parameter((a: any) => a.assignmentpattern((t: any) => t.left((i: any) => i.identifier((j: any) => j.name("d"))).right(
      (e: any) => e.identifier((i: any) => i.name("b")))))),
    (p: any) => p.memberexpression((m: any) => m.object(x).property((y: any) => y.identifier((i: any) => i.name("y"))))])), ["x"]],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.identifier((i: any) => i)]).body(x)), ["x"]], // a name not written binds none
  ] as [unknown, string[]][]) {
    assert(same([...SY.free(ts(spec), TS.DIALECT)].sort(), free));
  }
}

<a id="DIA-22"></a>
## DIA-22 · TypeScript5 printed as TypeScript writes it: every kind, its types, patterns and JSX, by precedence, as an expression statement holds it, a literal in its spelling or its value's

In [ ]:
{
  const ts = (spec: unknown): any => term(spec, TS.DIALECT);
  const shown = (spec: unknown): string => TSText.ToText(ts(spec)); // a TypeScript5 term's source
  // Specs of terms, types and patterns, written through the builders, each in any union whose branches it names
  const [x, y, z] = ["x", "y", "z"].map((n) => (e: any) => e.identifier((i: any) => i.name(n)));
  const one = (e: any) => e.literal((r: any) => r.value((n: any) => n.as_scalar(1.0)));
  const number = (t: any) => t.tsnumberkeyword((k: any) => k);
  const string = (t: any) => t.tsstringkeyword((k: any) => k);
  const [T_, U_] = ["T", "U"].map((n) => (t: any) => t.tstypereference((r: any) => r.typeName((m: any) => m.identifier((i: any) => i.name(n)))));
  const empty = (e: any) => e.objectexpression((r: any) => r); // `{}`
  // A number as JavaScript's `String(number)` writes it, a string in its quote with its escapes
  assert(same([NaN, Infinity, -Infinity, 0.0, -0.0, 100.0, 1.5, 0.001, 1e21, 1.5e-7, 5e-7, -2.5, 123456789012345680000.0, 1e-6]
    .map(TSText.number), ["NaN", "Infinity", "-Infinity", "0", "0", "100", "1.5", "0.001", "1e+21", "1.5e-7", "5e-7", "-2.5",
    "123456789012345680000", "0.000001"]));
  assert(TSText.string("it's \"x\"\\\n\x01\x7f\u2028é", "\"") === "\"it's \\\"x\\\"\\\\\\n\\x01\\x7f\\u2028é\"");
  assert(TSText.string("it's", "'") === "'it\\'s'");
  for (const [spec, text] of [
    // literals: a value's default spelling, else its text
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(true))), "true"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(false))), "false"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(5n))), "5n"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(1.5))), "1.5"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar("a\n"))), '"a\\n"'],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar("a")).quote("'")), "'a'"],
    [(u: any) => u.literal((r: any) => r.value((n: any) => n.as_scalar(16.0)).text("0x10")), "0x10"],
    [(u: any) => u.binaryexpression((r: any) => r.left((e: any) => e.literal((li: any) => li.value((n: any) => n.as_scalar(-5n))))
        .operator("**").right(one)), "(-5n) ** 1"],  // a negative value's precedence, a unary operator's
    [(u: any) => u.constant((r: any) => r.singleton("null").text("null")), "null"],
    [(u: any) => u.constant((r: any) => r.singleton("null")), "null"],
    [(u: any) => u.constant((r: any) => r.pattern("a+").flags("g")), "/a+/g"],
    [(u: any) => u.constant((r: any) => r.pattern("a")), "/a/"],
    // names and templates
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.identifier((i: any) => i.decorators([(d: any) => d.expression(x)])
        .name("y").optional(true).typeAnnotation((a: any) => a.typeAnnotation(number))), (p: any) => p.identifier((i: any) => i)])),
     "(@x y?: number, ) => {}"],  // a parameter's decorators, `?` and type; one of no name, as held
    [(u: any) => u.memberexpression((r: any) => r.object(x).property((p: any) => p.privateidentifier((i: any) => i.name("p")))), "x.#p"],
    [(u: any) => u.templateliteral((r: any) => r), "``"],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.tail(true)])), "``"],
    [(u: any) => u.templateliteral((r: any) => r.quasis([(q: any) => q.raw("a"), (q: any) => q.tail(true)]).expressions([x])), "`a${x}`"],
    [(u: any) => u.taggedtemplateexpression((r: any) => r.tag(x).typeArguments((a: any) => a.params([number])).quasi(
        (q: any) => q.quasis([(e: any) => e.raw("a").tail(true)]))), "x<number>`a`"],
    [(u: any) => u.parenthesizedexpression((r: any) => r.expression((e: any) => e.sequenceexpression((s: any) => s.expressions([x, y])))), "(x, y)"],
    // displays
    [(u: any) => u.arrayexpression((r: any) => r), "[]"],
    [(u: any) => u.arrayexpression((r: any) => r.elements([(e: any) => e.typescript5((t: any) => t.elision((h: any) => h)),
                                                       (e: any) => e.typescript5(x), (e: any) => e.typescript5((t: any) => t.elision((h: any) => h))])),
     "[, x,,]"],  // a hole as nothing, a last hole kept
    [empty, "({})"],  // an object, as a statement holds it
    [(u: any) => u.objectexpression((r: any) => r.properties([
        (p: any) => p.property((q: any) => q.key(x).shorthand(true).value(x)),
        (p: any) => p.property((q: any) => q.key(y).shorthand(true).value((v: any) => v.assignmentpattern((a: any) => a.left(y).right(one)))),
        (p: any) => p.property((q: any) => q.propertyKind("get").key(z).value((v: any) => v.functionexpression((f: any) => f))),
        (p: any) => p.property((q: any) => q.propertyKind("set").key(z).value((v: any) => v.functionexpression((f: any) => f.params([x])))),
        (p: any) => p.property((q: any) => q.method(true).key(x).value((v: any) => v.functionexpression((f: any) => f.isAsync(true).generator(true)))),
        (p: any) => p.property((q: any) => q.method(true).optional(true).key(x).value((v: any) => v.functionexpression((f: any) => f))),
        (p: any) => p.property((q: any) => q.method(true).key(x).value((v: any) => v.tsemptybodyfunctionexpression(
            (f: any) => f.returnType((a: any) => a.typeAnnotation(number))))),
        (p: any) => p.property((q: any) => q.computed(true).key(x).value((v: any) => v.sequenceexpression((s: any) => s.expressions([x, y])))),
        (p: any) => p.spreadelement((s: any) => s.argument(x))])),
     "({ x, y = 1, get z() {}, set z(x) {}, async *x() {}, x?() {}, x(): number, [x]: (x, y), ...x })"],
    // functions and classes: a body of statements is written `{}`
    [(u: any) => u.functionexpression((r: any) => r), "(function() {})"],
    [(u: any) => u.sequenceexpression((s: any) => s.expressions([x, (e: any) => e.functionexpression((r: any) => r.isAsync(true).generator(true)
        .id((i: any) => i.name("f")).typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
            (p: any) => p.identifier((i: any) => i.name("a").typeAnnotation((a: any) => a.typeAnnotation(T_)))]).returnType(
            (a: any) => a.typeAnnotation(T_)))])), "x, async function* f<T>(a: T): T {}"],
    [(u: any) => u.tsemptybodyfunctionexpression((r: any) => r.isAsync(true).generator(true)), "async *()"],
    [(u: any) => u.tsemptybodyfunctionexpression((r: any) => r), "()"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))]))
        .params([x])), "<T,>(x) => {}"],  // a lone type parameter, as a .tsx file reads it
    [(u: any) => u.arrowfunctionexpression((r: any) => r.isAsync(true).typeParameters((d: any) => d.params([
        (p: any) => p.name((i: any) => i.name("T")).constraint(U_)])).body(x)), "async <T extends U>() => x"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.typeParameters((d: any) => d.params([
        (p: any) => p.name((i: any) => i.name("T")), (p: any) => p.name((i: any) => i.name("U"))])).body(x)), "<T, U>() => x"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.body(empty)), "() => ({})"],  // an object body, parenthesized
    [(u: any) => u.arrowfunctionexpression((r: any) => r.body((b: any) => b.memberexpression((m: any) => m.object(empty).property(x)))),
     "() => ({}.x)"],
    [(u: any) => u.arrowfunctionexpression((r: any) => r.body((b: any) => b.sequenceexpression((s: any) => s.expressions([x, y])))), "() => (x, y)"],
    [(u: any) => u.classexpression((r: any) => r), "(class {})"],
    [(u: any) => u.sequenceexpression((s: any) => s.expressions([x, (e: any) => e.classexpression((r: any) => r.decorators([(d: any) => d.expression(x)])
        .id((i: any) => i.name("K")).typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).superClass(x)
        .superTypeArguments((a: any) => a.params([number])).implements([(c: any) => c.expression(y).typeArguments((a: any) => a.params([T_])),
                                                                       (c: any) => c.expression(z)]))])),
     "x, @x class K<T> extends x<number> implements y<T>, z {}"],
    [(u: any) => u.sequenceexpression((s: any) => s.expressions([x, (e: any) => e.classexpression((r: any) => r.superClass((c: any) => c.callexpression(
        (f: any) => f.callee(x))))])), "x, class extends x() {}"],
    [(u: any) => u.metaproperty((r: any) => r.meta((i: any) => i.name("new")).property((i: any) => i.name("target"))), "new.target"],
    // operations
    [(u: any) => u.memberexpression((r: any) => r.object(one).property(x)), "(1).x"],  // `1.x` would be a number
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.literal((li: any) => li.value((n: any) => n.as_scalar(1.5)))).property(x)), "1.5.x"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.literal((li: any) => li.value((n: any) => n.as_scalar(1000.0)).text("1_000"))).property(x)),
     "(1_000).x"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.thisexpression((h: any) => h)).computed(true).optional(true).property(y)), "this?.[y]"],
    [(u: any) => u.memberexpression((r: any) => r.object(x).computed(true).property((p: any) => p.sequenceexpression((s: any) => s.expressions([x, y])))),
     "x[x, y]"],
    [(u: any) => u.chainexpression((r: any) => r.expression((e: any) => e.memberexpression((m: any) => m.object(x).optional(true).property(y)))), "x?.y"],
    [(u: any) => u.callexpression((r: any) => r.callee(x).optional(true).typeArguments((a: any) => a.params([number])).arguments([
        y, (e: any) => e.sequenceexpression((s: any) => s.expressions([x, y]))])), "x?.<number>(y, (x, y))"],
    [(u: any) => u.newexpression((r: any) => r.callee((c: any) => c.callexpression((f: any) => f.callee(x)))), "new (x())()"],
    [(u: any) => u.newexpression((r: any) => r.callee((c: any) => c.memberexpression((m: any) => m.object((o: any) => o.tsnonnullexpression(
        (n: any) => n.expression(x))).property(y)))), "new (x!.y)()"],  // a callee that calls, in parentheses
    [(u: any) => u.newexpression((r: any) => r.callee((c: any) => c.memberexpression((m: any) => m.object(x).property(y))).typeArguments(
        (a: any) => a.params([T_])).arguments([z])), "new x.y<T>(z)"],
    [(u: any) => u.importexpression((r: any) => r.phase("defer").source(x)), "import.defer(x)"],
    [(u: any) => u.importexpression((r: any) => r.source(x).options(y)), "import(x, y)"],
    [(u: any) => u.updateexpression((r: any) => r.operator("++").prefix(true).argument(x)), "++x"],
    [(u: any) => u.updateexpression((r: any) => r.operator("--").argument((a: any) => a.binaryexpression((b: any) => b.left(x).operator("+").right(y)))),
     "(x + y)--"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("void").argument(x)), "void x"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("-").argument((a: any) => a.unaryexpression((b: any) => b.operator("-").argument(x)))), "- -x"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("+").argument((a: any) => a.literal((li: any) => li.value((n: any) => n.as_scalar(1.0)).text("+1")))), "+ +1"],
    [(u: any) => u.unaryexpression((r: any) => r.operator("!").argument((a: any) => a.binaryexpression((b: any) => b.left(x).operator("in").right(y)))), "!(x in y)"],
    [(u: any) => u.awaitexpression((r: any) => r.argument(x)), "await x"],
    // binary and logical operators by precedence: `**`'s left, `??` against `||` and `&&`
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.binaryexpression((b: any) => b.left(x).operator("+").right(y))).operator("*").right(
        (a: any) => a.binaryexpression((b: any) => b.left(x).operator("-").right(y)))), "(x + y) * (x - y)"],
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.binaryexpression((b: any) => b.left(x).operator("**").right(y))).operator("**").right(
        (a: any) => a.binaryexpression((b: any) => b.left(x).operator("**").right(y)))), "(x ** y) ** x ** y"],
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.awaitexpression((b: any) => b.argument(x))).operator("**").right(y)), "(await x) ** y"],
    [(u: any) => u.binaryexpression((r: any) => r.left((a: any) => a.tstypeassertion((b: any) => b.typeAnnotation(number).expression(x))).operator("**").right(y)),
     "(<number>x) ** y"],
    [(u: any) => u.logicalexpression((r: any) => r.left((a: any) => a.logicalexpression((b: any) => b.left(x).operator("??").right(y))).operator("||").right(
        (a: any) => a.logicalexpression((b: any) => b.left(x).operator("&&").right(y)))), "(x ?? y) || x && y"],
    [(u: any) => u.logicalexpression((r: any) => r.left(x).operator("??").right((a: any) => a.logicalexpression((b: any) => b.left(y).operator("||").right(z)))),
     "x ?? (y || z)"],
    [(u: any) => u.logicalexpression((r: any) => r.left((a: any) => a.logicalexpression((b: any) => b.left(x).operator("??").right(y))).operator("??").right(z)),
     "x ?? y ?? z"],
    [(u: any) => u.conditionalexpression((r: any) => r.test((t: any) => t.conditionalexpression((c: any) => c.test(x).consequent(y).alternate(z)))
        .consequent((c: any) => c.assignmentexpression((a: any) => a.left(x).operator("=").right(y))).alternate(z)), "(x ? y : z) ? x = y : z"],
    // an assignment's target, a pattern: a term it may be, or a destructuring
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.memberexpression((m: any) => m.object(x).property(y))).operator("+=").right(one)),
     "x.y += 1"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.arraypattern((a: any) => a.elements([x, (e: any) => e.elision((h: any) => h),
        (e: any) => e.restelement((s: any) => s.argument(y))]))).operator("=").right(z)), "[x,, ...y] = z"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.objectpattern((o: any) => o)).operator("=").right(z)), "({} = z)"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.tsasexpression((a: any) => a.expression(x).typeAnnotation(
        (k: any) => k.tsanykeyword((h: any) => h)))).operator("=").right(z)), "(x as any) = z"],
    [(u: any) => u.assignmentexpression((r: any) => r.left((t: any) => t.parenthesizedexpression((a: any) => a.expression(x))).operator("??=").right(z)),
     "(x) ??= z"],
    [(u: any) => u.yieldexpression((r: any) => r), "yield"],
    [(u: any) => u.yieldexpression((r: any) => r.argument((a: any) => a.sequenceexpression((s: any) => s.expressions([x, y])))), "yield (x, y)"],
    [(u: any) => u.tsasexpression((r: any) => r.expression((e: any) => e.tsasexpression((a: any) => a.expression(x).typeAnnotation(T_))).typeAnnotation(U_)),
     "x as T as U"],
    [(u: any) => u.tssatisfiesexpression((r: any) => r.expression((e: any) => e.binaryexpression((b: any) => b.left(x).operator("==").right(y)))
        .typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([number, string])))), "(x == y) satisfies number | string"],
    [(u: any) => u.tsinstantiationexpression((r: any) => r.expression((e: any) => e.tsnonnullexpression((n: any) => n.expression(x))).typeArguments(
        (a: any) => a.params([T_]))), "x!<T>"],
    // patterns, as parameters
    [(u: any) => u.arrowfunctionexpression((r: any) => r.params([
        (p: any) => p.arraypattern((a: any) => a.decorators([(d: any) => d.expression(x)]).elements([x, (e: any) => e.elision((h: any) => h)]).optional(true)
            .typeAnnotation((t: any) => t.typeAnnotation((k: any) => k.tsarraytype((n: any) => n.elementType(number))))),
        (p: any) => p.objectpattern((o: any) => o.properties([(q: any) => q.property((s: any) => s.key(x).shorthand(true).value(x)),
            (q: any) => q.property((s: any) => s.key(y).value((v: any) => v.assignmentpattern((a: any) => a.left(z).right(one)))),
            (q: any) => q.restelement((s: any) => s.argument(z))])),
        (p: any) => p.objectpattern((o: any) => o.optional(true)),
        (p: any) => p.assignmentpattern((a: any) => a.left((t: any) => t.identifier((i: any) => i.name("x").typeAnnotation((b: any) => b.typeAnnotation(number))))
            .right(one)),
        (p: any) => p.restelement((s: any) => s.argument(y).typeAnnotation((t: any) => t.typeAnnotation((k: any) => k.tsarraytype((n: any) => n.elementType(T_))))),
        (p: any) => p.tsparameterproperty((s: any) => s.decorators([(d: any) => d.expression(x)]).accessibility("private").static(true).override(true)
            .readonly(true).parameter(z))])),
     "(@x [x,,]?: number[], { x, y: z = 1, ...z }, {}?, x: number = 1, ...y: T[], @x private static override readonly z) => {}"],
    // types, as `x as <type>`
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstupletype((n: any) => n.elementTypes([
        ...["tsanykeyword", "tsunknownkeyword", "tsnumberkeyword", "tsbigintkeyword", "tsbooleankeyword", "tsstringkeyword",
          "tssymbolkeyword", "tsobjectkeyword", "tsneverkeyword", "tsvoidkeyword", "tsundefinedkeyword", "tsnullkeyword",
          "tsintrinsickeyword", "tsthistype"].map((keyword) => (k: any) => k[keyword]((h: any) => h))])))),
     "x as [any, unknown, number, bigint, boolean, string, symbol, object, never, void, undefined, null, intrinsic, this]"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tstypereference((q: any) => q.typeName((m: any) => m.tsqualifiedname((a: any) => a.left((l: any) => l.tsqualifiedname(
            (b: any) => b.left(x).right((i: any) => i.name("y")))).right((i: any) => i.name("z")))).typeArguments((a: any) => a.params([T_]))),
        (k: any) => k.tstypereference((q: any) => q.typeName((m: any) => m.thisexpression((h: any) => h))),
        (k: any) => k.tsparenthesizedtype((q: any) => q.typeAnnotation(number)),
        (k: any) => k.tsliteraltype((q: any) => q.literal((li: any) => li.literal((v: any) => v.value((n: any) => n.as_scalar("s"))))),
        (k: any) => k.tsliteraltype((q: any) => q.literal((li: any) => li.unaryexpression((v: any) => v.operator("-").argument(one)))),
        (k: any) => k.tsliteraltype((q: any) => q.literal((li: any) => li.templateliteral((v: any) => v.quasis([(e: any) => e.raw("a").tail(true)])))),
        (k: any) => k.tstemplateliteraltype((q: any) => q.quasis([(e: any) => e.raw("a"), (e: any) => e.tail(true)]).types([number])),
        (k: any) => k.tstemplateliteraltype((q: any) => q), (k: any) => k.tstemplateliteraltype((q: any) => q.quasis([(e: any) => e.tail(true)]))])))),
     'x as x.y.z<T> | this | (number) | "s" | -1 | `a` | `a${number}` | `` | ``'],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsintersectiontype((n: any) => n.types([
        (k: any) => k.tsarraytype((a: any) => a.elementType((e: any) => e.tsuniontype((q: any) => q.types([number, string])))),
        (k: any) => k.tsuniontype((q: any) => q.types([number, (e: any) => e.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation(
            (v: any) => v.tsvoidkeyword((h: any) => h))))])),
        (k: any) => k.tstupletype((q: any) => q.elementTypes([
            (e: any) => e.tsnamedtuplemember((m: any) => m.label((i: any) => i.name("a")).elementType(number)),
            (e: any) => e.tsnamedtuplemember((m: any) => m.label((i: any) => i.name("b")).optional(true).elementType(string)),
            (e: any) => e.tsoptionaltype((o: any) => o.typeAnnotation(number)),
            (e: any) => e.tsresttype((o: any) => o.typeAnnotation((a: any) => a.tsarraytype((b: any) => b.elementType(T_))))])),
        (k: any) => k.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
            (p: any) => p.identifier((i: any) => i.name("a").typeAnnotation((a: any) => a.typeAnnotation(T_)))]).returnType(
            (a: any) => a.typeAnnotation(T_))),
        (k: any) => k.tsconstructortype((f: any) => f.abstract(true).returnType((a: any) => a.typeAnnotation(T_))),
        (k: any) => k.tsconstructortype((f: any) => f)])))),
     "x as (number | string)[] & (number | (() => void)) & [a: number, b?: string, number?, ...T[]] & (<T>(a: T) => T) & (abstract new () => T) & (new ())"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tstypeliteral((q: any) => q),
        (k: any) => k.tstypeliteral((q: any) => q.members([
            (m: any) => m.tspropertysignature((s: any) => s.readonly(true).key(x).optional(true).typeAnnotation((a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tsmethodsignature((s: any) => s.computed(true).key(y).optional(true).returnType((a: any) => a.typeAnnotation(
                (v: any) => v.tsvoidkeyword((h: any) => h)))),
            (m: any) => m.tsmethodsignature((s: any) => s.methodKind("get").key(z).returnType((a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tsmethodsignature((s: any) => s.methodKind("method").key(z)),
            (m: any) => m.tscallsignaturedeclaration((s: any) => s.returnType((a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tsconstructsignaturedeclaration((s: any) => s.params([x])),
            (m: any) => m.tsindexsignature((s: any) => s.accessibility("public").static(true).readonly(true).parameters([
                (p: any) => p.identifier((i: any) => i.name("k").typeAnnotation((a: any) => a.typeAnnotation(string)))]).typeAnnotation(
                (a: any) => a.typeAnnotation(T_))),
            (m: any) => m.tspropertysignature((s: any) => s.key((k: any) => k.literal((li: any) => li.value((n: any) => n.as_scalar("k"))))),
            (m: any) => m.tsindexsignature((s: any) => s.parameters([x]))]))])))),
     'x as {} | { readonly x?: T; [y]?(): void; get z(): T; z(); (): T; new (x); public static readonly [k: string]: T; "k"; [x]; }'],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tstypeliteral((q: any) => q.members([
        (m: any) => m.comment((c: any) => c.text(" first").trailing(true)),
        (m: any) => m.tspropertysignature((s: any) => s.key(x)),
        (m: any) => m.comment((c: any) => c.text(" of x").trailing(true)),
        (m: any) => m.comment((c: any) => c.block(true).text(" y ")),
        (m: any) => m.comment((c: any) => c)])))),
     "x as {\n    // first\n    x; // of x\n    /* y */\n    //\n}"],  // with comments, its members' lines
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tsmappedtype((m: any) => m.readonly("+readonly").key((i: any) => i.name("K")).constraint(T_).nameType(U_).optional("?").typeAnnotation(T_)),
        (k: any) => k.tsmappedtype((m: any) => m.key((i: any) => i.name("K")).constraint(T_)),
        (k: any) => k.tsindexedaccesstype((a: any) => a.objectType((o: any) => o.tsuniontype((q: any) => q.types([T_, U_]))).indexType(
            (i: any) => i.tstypeoperator((o: any) => o.operator("keyof").typeAnnotation(T_)))),
        (k: any) => k.tstypeoperator((o: any) => o.operator("unique").typeAnnotation((s: any) => s.tssymbolkeyword((h: any) => h))),
        (k: any) => k.tstypeoperator((o: any) => o.operator("readonly").typeAnnotation((a: any) => a.tsuniontype((q: any) => q.types([T_, U_])))),
        (k: any) => k.tstypeoperator((o: any) => o.operator("keyof")),
        (k: any) => k.tstypequery((q: any) => q.exprName(x).typeArguments((a: any) => a.params([T_]))),
        (k: any) => k.tstypequery((q: any) => q.exprName((e: any) => e.tsimporttype((i: any) => i.source((s: any) => s.literal(
            (li: any) => li.value((n: any) => n.as_scalar("m"))))))),
        (k: any) => k.tsimporttype((i: any) => i.source((s: any) => s.literal((li: any) => li.value((n: any) => n.as_scalar("m")).quote("'"))).options(
            (o: any) => o.objectexpression((e: any) => e.properties([(p: any) => p.property((q: any) => q.key((w: any) => w.identifier((j: any) => j.name("with")))
                .value(empty))]))).qualifier((q: any) => q.identifier((j: any) => j.name("A"))).typeArguments((a: any) => a.params([T_])))])))),
     "x as { +readonly [K in T as U]?: T } | { [K in T] } | (T | U)[keyof T] | unique symbol | readonly (T | U) | keyof | typeof x<T> | "
     + "typeof import(\"m\") | import('m', { with: {} }).A<T>"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsconditionaltype((c: any) => c
        .checkType((k: any) => k.tsuniontype((q: any) => q.types([T_, U_])))
        .extendsType((k: any) => k.tsarraytype((a: any) => a.elementType((e: any) => e.tsinfertype((i: any) => i.typeParameter(
            (p: any) => p.name((n: any) => n.name("V")).constraint(string)))))).trueType((k: any) => k.tsconditionaltype((d: any) => d
                .checkType(T_).extendsType((e: any) => e.tsconditionaltype((f: any) => f.checkType(T_).extendsType(U_).trueType(T_).falseType(U_)))
                .trueType(T_).falseType(U_))).falseType((k: any) => k.tsconditionaltype((d: any) => d.checkType(T_)
                    .extendsType((e: any) => e.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation(
                        (g: any) => g.tsconditionaltype((h: any) => h.checkType(T_).extendsType(U_).trueType(T_).falseType(U_))))))
                    .trueType((e: any) => e.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation(T_)))).falseType(
                    (e: any) => e.tsconditionaltype((h: any) => h.checkType(T_).extendsType((g: any) => g.tsfunctiontype((f: any) => f)).trueType(T_)
                        .falseType(U_)))))))),
     "x as T | U extends (infer V extends string)[] ? T extends (T extends U ? T : U) ? T : U : "
     + "T extends (() => T extends U ? T : U) ? () => T : T extends () ? T : U"],
    [(u: any) => u.tsasexpression((r: any) => r.expression(x).typeAnnotation((t: any) => t.tsuniontype((n: any) => n.types([
        (k: any) => k.tsfunctiontype((f: any) => f.params([x]).returnType((a: any) => a.typeAnnotation((p: any) => p.tstypepredicate((q: any) => q
            .asserts(true).parameterName(x).typeAnnotation((b: any) => b.typeAnnotation(T_)))))),
        (k: any) => k.tsfunctiontype((f: any) => f.returnType((a: any) => a.typeAnnotation((p: any) => p.tstypepredicate((q: any) => q.parameterName(
            (m: any) => m.tsthistype((h: any) => h)))))),
        (k: any) => k.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.const(true).isIn(true).isOut(true).name(
            (i: any) => i.name("T")).default(U_)])))])))),
     "x as ((x) => asserts x is T) | (() => this) | (<const in out T = U>())"],
    // JSX, as written
    [(u: any) => u.jsxelement((r: any) => r.openingElement((o: any) => o.name((n: any) => n.jsxmemberexpression((m: any) => m.object(
        (b: any) => b.jsxidentifier((i: any) => i.name("a"))).property((i: any) => i.name("b")))).typeArguments((a: any) => a.params([T_])).attributes([
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxnamespacedname((s: any) => s.namespace((i: any) => i.name("x")).name(
                (i: any) => i.name("y"))))),
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("z"))).value((v: any) => v.jsxexpressioncontainer(
                (c: any) => c.expression(one)))),
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("e"))).value((v: any) => v.jsxelement(
                (c: any) => c.openingElement((o: any) => o.name((n: any) => n.jsxidentifier((i: any) => i.name("b"))).selfClosing(true))))),
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("f"))).value((v: any) => v.jsxfragment((c: any) => c))),
            (a: any) => a.jsxspreadattribute((s: any) => s.argument((e: any) => e.sequenceexpression((q: any) => q.expressions([x, y]))))]))
        .children([(c: any) => c.jsxtext((t: any) => t.raw("hi ")), (c: any) => c.jsxtext((t: any) => t), (c: any) => c.jsxspreadchild((s: any) => s.expression(x)),
                   (c: any) => c.jsxelement((e: any) => e.openingElement((o: any) => o.name((n: any) => n.jsxidentifier((i: any) => i)))),
                   (c: any) => c.jsxexpressioncontainer((e: any) => e.expression((v: any) => v.jsxemptyexpression((h: any) => h)))])
        .closingElement((c: any) => c.name((n: any) => n.jsxmemberexpression((m: any) => m.object((b: any) => b.jsxidentifier((i: any) => i.name("a")))
            .property((i: any) => i.name("b")))))),
     "<a.b<T> x:y z={1} e=<b /> f=<></> {...(x, y)}>hi {...x}<></>{}</a.b>"],  // an element without its closing one closes its name
    // as an expression statement holds it: parenthesized where it would start with `{`, `function`, `class` or `let [`
    [(u: any) => u.memberexpression((r: any) => r.object(empty).property(x)), "({}.x)"],
    [(u: any) => u.binaryexpression((r: any) => r.left(empty).operator("+").right(x)), "({} + x)"],
    [(u: any) => u.callexpression((r: any) => r.callee((c: any) => c.functionexpression((f: any) => f))), "(function() {}())"],
    [(u: any) => u.taggedtemplateexpression((r: any) => r.tag((c: any) => c.classexpression((f: any) => f)).quasi(
      (q: any) => q.quasis([(e: any) => e.tail(true)]))), "(class {}``)"],
    [(u: any) => u.conditionalexpression((r: any) => r.test(empty).consequent(x).alternate(y)), "({} ? x : y)"],
    [(u: any) => u.tssatisfiesexpression((r: any) => r.expression(empty).typeAnnotation(T_)), "({} satisfies T)"],
    [(u: any) => u.updateexpression((r: any) => r.operator("++").argument(empty)), "({}++)"],
    [(u: any) => u.updateexpression((r: any) => r.operator("++").prefix(true).argument(empty)), "++{}"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.identifier((i: any) => i.name("let"))).computed(true).property(one)), "(let[1])"],
    [(u: any) => u.memberexpression((r: any) => r.object((e: any) => e.identifier((i: any) => i.name("let"))).property(x)), "let.x"],
  ] as [unknown, string][]) {
    const found = shown(spec);
    assert(found === text, `${Repr.repr(found)} != ${Repr.repr(text)}`);
  }
  raises(ValueError, () => shown((u: any) => u.binaryexpression((r: any) => r.left(x).operator("<>").right(y))),
    "TypeScript5 binaryexpression has no operator '<>'"); // an operator outside its vocabulary
}